# UCF101 Robust Text Watermark — Final Corrected Notebook

Notebook ini memperbaiki penyebab utama teks ekstraksi berubah acak setelah kompresi
H.264, H.265, dan neural codec.

Perubahan yang wajib untuk validitas eksperimen:

1. Decoder dilatih dengan **payload acak yang berubah per clip**, bukan terus-menerus
   diberi jawaban tetap `"sabila"`. Ini mencegah decoder menghafal keluaran.
2. Satu payload dipakai konsisten pada seluruh frame dalam satu clip sehingga noise
   H.264/H.265 melihat urutan temporal yang realistis.
3. Payload produksi `"sabila"` memakai repetition ECC 3x dan soft-voting antarsalinan.
4. Dataset dibagi berdasarkan **group UCF101 (`gXX`)**; group evaluasi tidak pernah
   dipakai untuk training.
5. Evaluasi otomatis dihentikan bila negative control atau random-payload sanity test
   gagal. Notebook tidak lagi melaporkan hasil palsu dari decoder yang collapse.
6. Neural codec menyimpan latent terkuantisasi (`.npz`) sebagai artefak kompresi;
   file rekonstruksi hanya dipakai untuk ekstraksi dan pengukuran kualitas.

Jalankan dari atas dengan **Runtime → Run all**. Jangan memakai checkpoint lama dari
versi fine-tune target tetap; notebook ini memakai nama checkpoint baru.


# [TAMBAHAN V2] Catatan Perbaikan — dibaca dulu sebelum menjalankan

Notebook ASLI Anda **tidak diubah/dihapus sama sekali** di bawah ini — setiap sel lama
(kode maupun markdown) masih persis sama isinya. Perbaikan ditambahkan sebagai **sel BARU**
yang disisipkan di titik-titik strategis (ditandai `[FIX]` di judul & komentarnya), sehingga
saat Anda menjalankan notebook dari atas ke bawah seperti biasa, perbaikan otomatis aktif.

**Akar masalah** (dari audit sebelumnya): watermark gagal terbaca bahkan SEBELUM video
dikompresi (BER lossless ~0,30; ~97% output decoder nyaris identik di semua video berbeda —
tanda decoder tidak benar-benar membaca sinyal, hanya menebak bias hampir konstan). Jadi
perbaikan difokuskan ke lapisan Encoder-Decoder & payload, bukan ke kompresinya.

## Daftar sel baru yang ditambahkan (semua bertanda `[FIX]`)

1. Setelah **Bagian 2** (Konfigurasi) — `[FIX #2]` ECC (repetition code): payload teks
   "sabila" (48 bit) diulang 3x menjadi 144 bit total, didekode dengan majority-vote agar
   tahan terhadap sejumlah bit yang salah.
2. Setelah **Bagian 4** (Arsitektur) — `[FIX #3]` Encoder/Decoder V2: kapasitas dinaikkan
   (channels 64→80, +1 blok residual) dan kekuatan residual watermark dinaikkan dari 0.10
   ke 0.18 (anggaran sinyal lama terlalu ketat untuk payload 144-bit).
3. Setelah **Bagian 5** (Training) — `[FIX #1, #4, #5]` Tahap fine-tune tambahan: watermark
   yang dipakai untuk training di tahap ini SAMA PERSIS dengan watermark target ("sabila"),
   dipakai konsisten ke SELURUH frame di setiap batch (meniru kondisi nyata), langsung di
   bawah noise H.264/H.265 penuh — memaksimalkan reliabilitas untuk watermark yang benar-benar
   akan dipakai.
4. Setelah **Bagian 6** (Penyisipan) — `[FIX]` Menyisipkan ULANG watermark dengan bit ECC
   yang benar (Bagian 6 asli sempat menyisipkan dengan bit versi lama/tanpa ECC terlebih
   dahulu; sel ini menimpanya dengan versi yang benar sebelum lanjut ke pengujian kompresi).
5. Setelah **Bagian 8** (Deteksi) — `[FIX]` Mencetak TEKS hasil ekstraksi (bukan cuma bit
   mentah) memakai decode ECC majority-vote — ini jawaban langsung untuk tujuan Anda: melihat
   apakah teks "sabila" berhasil dibaca kembali setelah video dikompresi.
6. Setelah **Bagian 8C** (Analisis per bit) — `[FIX]` Analisis tambahan per salinan ECC +
   berapa persen sampel yang teksnya berhasil dipulihkan PERSIS lewat majority-vote.

## Catatan penting
- Karena `WATERMARK_LENGTH` di-override dari 64 menjadi 144 bit, tabel "jenis bit" (payload
  vs padding) di **Bagian 8C asli** menjadi tidak akurat lagi (bit ke-49–144 sekarang berisi
  SALINAN ULANG payload, bukan padding nol) — ini murni label kosmetik di tabel lama; sel
  tambahan di poin 6 di atas memberi analisis yang benar sebagai gantinya.
- **WAJIB**: setelah menjalankan sel training tambahan (poin 3), cek dulu **Bagian 8B** —
  pastikan `bit_acc_lossless` tinggi dan teks cocok "sabila" — sebelum menghabiskan waktu
  menguji ketahanan kompresi di Bagian 7–8.


ya allah.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1rsY4K4dP9jM97QgIIwHVxUndoNDMr6BX

# DETEKSI VIDEO WATERMARKING TAHAN KOMPRESI TINGGI DENGAN NEURAL CODEC SECARA SISTEM OTOMATIS

**Notebook Google Colab — Implementasi Skripsi**

Notebook ini mengimplementasikan seluruh alur penelitian sesuai flowchart:

1. Input dataset video asli (dari **Google Drive**, bukan YouTube/`yt-dlp`)
2. Preprocessing video (resize, normalisasi, ekstraksi frame)
3. Penyisipan invisible watermark — **Neural Watermark Encoder**
4. Video ber-watermark
5. Pengujian kompresi: **H.264**, **H.265**, **Neural Codec**
6. Video hasil kompresi
7. Deteksi watermark — **Neural Watermark Decoder**
8. Verifikasi otomatis (watermark asli vs hasil ekstraksi)
9. Keputusan: **Terdeteksi / Tidak Terdeteksi**
10. Evaluasi: **Watermark Recovery Rate (TPR), BER, PSNR, SSIM, Similarity Score**
11. Analisis & visualisasi hasil

> **Catatan penting:** Sumber video sepenuhnya menggunakan **Google Drive**. Tidak ada bagian kode yang bergantung pada `yt-dlp` atau URL YouTube, sehingga proses pengambilan video stabil dan tidak error seperti pada versi sebelumnya.

## Novelty Penelitian (untuk naskah jurnal SINTA 2)

### **Compression-Aware Confidence Score (CACS)** — lihat Bagian 10B

Penelitian ini mengusulkan **satu kebaruan**, yaitu **CACS**: sebuah *confidence score* tunggal
yang menggantikan keputusan berbasis ambang tetap pada lapisan verifikasi watermark.

Pada sistem watermarking konvensional, status *Terdeteksi / Tidak Terdeteksi* ditentukan oleh
**satu metrik saja** (biasanya BER atau Similarity Score) terhadap **ambang yang ditetapkan
manual**, sehingga (a) kondisi kanal kompresi diabaikan dan (b) angka ambangnya tidak memiliki
dasar statistik.

**CACS** menggabungkan lima metrik yang sudah tersedia pada pipeline — **BER, PSNR, SSIM,
Similarity Score, dan Compression Ratio** — menjadi **satu nilai confidence pada rentang [0, 1]**:

- setiap metrik dinormalisasi terhadap *acceptance anchor* dari jurnal bereputasi
  (PSNR 30 dB, SSIM 0,90, NC 0,75) sehingga nilai-terima literatur selalu jatuh tepat di skor 0,5;
- ambang BER **diturunkan secara statistik** dari uji binomial dengan target
  *false positive rate* $10^{-6}$ (bukan angka yang ditetapkan manual);
- penggabungan memakai **equal weighting** sesuai OECD/JRC *Handbook on Constructing
  Composite Indicators* (2008), karena tidak ada literatur yang memberi bobot gabungan baku
  untuk kelima metrik ini;
- ambang keputusan $	au = 0{,}50$ merupakan **konsekuensi konstruksi normalisasi**, bukan pilihan bebas;
- keputusan akhir memakai **aturan dua-syarat** agar metrik kualitas tidak dapat mengkompensasi
  ketiadaan bukti payload.

Seluruh arsitektur Deep Learning (Encoder, Decoder, Neural Codec), proses training, kompresi,
dan ekstraksi **tidak diubah sama sekali**. Kebaruan berada murni pada lapisan verifikasi.

---

### Catatan Perbaikan Teknis (agar training benar-benar bisa belajar)

Di luar novelty CACS, notebook ini memperbaiki 3 bug teknis yang menyebabkan training gagal / tidak konsisten dengan judul:

1. **Neural Codec sekarang benar-benar ikut dalam proses training watermark** (bukan hanya dipakai saat pengujian akhir), memakai *Straight-Through Estimator* agar kuantisasi lossy-nya tetap bisa dilatih dengan gradien.
2. Bug variabel yang tidak terdefinisi pada pelatihan Neural Codec (`all_frames_t`, `n`) sudah diperbaiki.
3. Ditambahkan augmentasi ringan (flip horizontal acak) pada batch training untuk membantu mengatasi jumlah dataset video yang terbatas.

## 1. Setup Environment & Mount Google Drive

Sel ini memasang Google Drive sebagai sumber video (menggantikan `yt-dlp`/YouTube) dan menginstal seluruh dependency yang dibutuhkan.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library pemrosesan video, deep learning, dataset, dan evaluasi.
!pip install -q "kagglehub>=0.4.3" opencv-python-headless scikit-image torch torchvision tqdm pandas matplotlib
!apt-get update -qq
!apt-get -y install ffmpeg > /dev/null 2>&1

print("Setup selesai. Google Drive terpasang di /content/drive")


## 2. Dataset UCF101, Path Proyek, dan Split Eksperimen

Dataset: <https://www.kaggle.com/datasets/abdallahwagih/ucf101-videos>

Struktur Google Drive:

```text
MyDrive/Order/order_20260827_213103/
├── main.ipynb
├── dataset/       # file .avi/.zip UCF101; otomatis diunduh jika belum ada
└── output/
    ├── watermarked/
    ├── compressed/
    ├── frames_tmp/
    ├── models/
    └── reports/
```

Urutan setup dataset:

1. Jika `.avi` sudah ada, langsung digunakan.
2. Jika belum ada `.avi` tetapi ada `.zip`, arsip diekstrak.
3. Jika keduanya tidak ada, dataset diunduh melalui KaggleHub.

Untuk mencegah *data leakage*, pembagian train/evaluation dilakukan berdasarkan nomor
group UCF101 (`gXX`), bukan sekadar membagi clip secara acak.


In [ ]:
import os
import random
import re
import zipfile
from pathlib import Path

import numpy as np
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/Order/order_20260827_213103')
DATASET_DIR = PROJECT_DIR / 'dataset'
OUTPUT_DIR = PROJECT_DIR / 'output'

WATERMARKED_DIR = OUTPUT_DIR / 'watermarked'
COMPRESSED_DIR = OUTPUT_DIR / 'compressed'
FRAMES_DIR = OUTPUT_DIR / 'frames_tmp'
MODELS_DIR = OUTPUT_DIR / 'models'
REPORTS_DIR = OUTPUT_DIR / 'reports'

for directory in (
    DATASET_DIR,
    OUTPUT_DIR,
    WATERMARKED_DIR,
    COMPRESSED_DIR,
    FRAMES_DIR,
    MODELS_DIR,
    REPORTS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Konfigurasi eksperimen yang aman untuk Google Colab.
# Naikkan batas video setelah pipeline lolos sanity test.
# ----------------------------------------------------------
FRAME_SIZE = (128, 128)
MAX_FRAMES_PER_VIDEO = 48
MAX_TRAIN_VIDEOS = 20
MAX_EVAL_VIDEOS = 10
EVAL_GROUP_FRACTION = 0.20
SEED = 42

WATERMARK_TEXT = "sabila"
ECC_REPEAT = 3
PAYLOAD_BITS = 8 * len(WATERMARK_TEXT)
WATERMARK_LENGTH = PAYLOAD_BITS * ECC_REPEAT

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# ----------------------------------------------------------
# Dataset: AVI -> ZIP -> KaggleHub.
# ----------------------------------------------------------
KAGGLE_DATASET_HANDLE = 'abdallahwagih/ucf101-videos'

def find_avi_files(root):
    return sorted(Path(root).rglob('*.avi'))


def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = (destination / member.filename).resolve()
            if member_path != destination and destination not in member_path.parents:
                raise RuntimeError(f"Path tidak aman di arsip {zip_path}: {member.filename}")
        archive.extractall(destination)


dataset_roots = [DATASET_DIR]
all_avi_paths = find_avi_files(DATASET_DIR)

if all_avi_paths:
    print(f"Dataset siap: ditemukan {len(all_avi_paths)} file AVI.")
else:
    zip_files = sorted(DATASET_DIR.rglob('*.zip'))
    if zip_files:
        print(f"AVI belum ada; mengekstrak {len(zip_files)} arsip ZIP...")
        for zip_path in zip_files:
            print(" -", zip_path.name)
            safe_extract_zip(zip_path, DATASET_DIR)
    else:
        print("AVI/ZIP belum ada; mengunduh UCF101 dari Kaggle...")
        import kagglehub
        downloaded_path = Path(
            kagglehub.dataset_download(
                KAGGLE_DATASET_HANDLE,
                output_dir=str(DATASET_DIR),
            )
        )
        if downloaded_path not in dataset_roots:
            dataset_roots.append(downloaded_path)
        print("KaggleHub selesai:", downloaded_path)

    all_avi_paths = sorted({
        path.resolve()
        for root in dataset_roots
        for path in find_avi_files(root)
    })

if not all_avi_paths:
    raise FileNotFoundError(
        f"Tidak ada .avi setelah setup dataset. Periksa isi {DATASET_DIR} "
        "dan autentikasi Kaggle bila unduhan meminta login."
    )

# ----------------------------------------------------------
# Split berdasarkan group UCF101: v_Action_gXX_cYY.avi
# Tidak ada group yang muncul di train sekaligus evaluation.
# ----------------------------------------------------------
GROUP_RE = re.compile(r'_g(\d+)_c\d+\.avi$', re.IGNORECASE)

def ucf_group(path):
    match = GROUP_RE.search(path.name)
    return int(match.group(1)) if match else None


valid_paths = [path for path in all_avi_paths if ucf_group(path) is not None]
if len(valid_paths) < 2:
    raise RuntimeError(
        "Nama file UCF101 tidak mengikuti pola v_Action_gXX_cYY.avi atau video terlalu sedikit."
    )

group_ids = sorted({ucf_group(path) for path in valid_paths})
split_rng = random.Random(SEED)
split_rng.shuffle(group_ids)

if len(group_ids) < 2:
    raise RuntimeError("Butuh minimal dua group UCF101 untuk train/evaluation terpisah.")

n_eval_groups = max(1, int(round(len(group_ids) * EVAL_GROUP_FRACTION)))
n_eval_groups = min(n_eval_groups, len(group_ids) - 1)
eval_groups = set(group_ids[:n_eval_groups])
train_groups = set(group_ids[n_eval_groups:])

train_candidates = [p for p in valid_paths if ucf_group(p) in train_groups]
eval_candidates = [p for p in valid_paths if ucf_group(p) in eval_groups]
split_rng.shuffle(train_candidates)
split_rng.shuffle(eval_candidates)

train_paths = train_candidates[:MAX_TRAIN_VIDEOS]
eval_paths = eval_candidates[:MAX_EVAL_VIDEOS]
if not train_paths or not eval_paths:
    raise RuntimeError("Split menghasilkan train/evaluation kosong; periksa dataset.")

selected_paths = train_paths + eval_paths
names = [path.name for path in selected_paths]
if len(names) != len(set(names)):
    raise RuntimeError("Nama file AVI duplikat ditemukan; gunakan nama UCF101 yang unik.")

VIDEO_PATHS = {path.name: path for path in selected_paths}
TRAIN_VIDEO_NAMES = [path.name for path in train_paths]
EVAL_VIDEO_NAMES = [path.name for path in eval_paths]
video_list = TRAIN_VIDEO_NAMES + EVAL_VIDEO_NAMES

assert set(train_groups).isdisjoint(eval_groups)
assert set(TRAIN_VIDEO_NAMES).isdisjoint(EVAL_VIDEO_NAMES)

print(
    f"Watermark: '{WATERMARK_TEXT}' | payload={PAYLOAD_BITS} bit | "
    f"ECC x{ECC_REPEAT} | output decoder={WATERMARK_LENGTH} bit"
)
print(f"Group train : {sorted(train_groups)}")
print(f"Group eval  : {sorted(eval_groups)}")
print(f"Video aktif : {len(TRAIN_VIDEO_NAMES)} train + {len(EVAL_VIDEO_NAMES)} evaluation")



## [PHASE-1 FIX] Payload & ECC — Satu Sumber Ground Truth

Payload produksi tetap **`"sabila"`**, tetapi konfigurasi teks, panjang payload, jumlah
repetition ECC, dan panjang output decoder sekarang didefinisikan **sekali** di Bagian 2.

Tujuan perubahan ini adalah menghindari state notebook yang sebelumnya sempat berganti dari
64 bit → 144 bit dan kemudian mendefinisikan ulang fungsi konversi di cell lain.

Training generalisasi **tidak menghafalkan `"sabila"`**. Model dilatih membaca payload acak;
`"sabila"` hanya menjadi payload produksi/evaluasi.


In [ ]:

print("=== [PHASE-1 FIX] ECC Repetition Code ===")
print(
    f"Payload '{WATERMARK_TEXT}' = {PAYLOAD_BITS} bit, diulang {ECC_REPEAT}x "
    f"-> {WATERMARK_LENGTH} bit."
)

def text_to_bits_ecc(text, repeat=ECC_REPEAT, bit_length=None):
    """Teks -> payload bit -> ulangi SELURUH payload sebanyak `repeat` kali."""
    raw = []
    for ch in text:
        raw.extend(int(b) for b in format(ord(ch), '08b'))

    if bit_length is None:
        bit_length = len(raw) * repeat

    bits = (raw * repeat)[:bit_length]
    bits += [0] * max(0, bit_length - len(bits))
    return bits, len(raw)


def ecc_majority_vote(bits, n_payload_bits=PAYLOAD_BITS, repeat=ECC_REPEAT):
    """Kembalikan payload asli setelah majority-vote antar salinan ECC."""
    arr = np.asarray(bits).reshape(-1)
    if arr.size < n_payload_bits * repeat:
        raise ValueError(
            f"Bit ECC kurang: butuh {n_payload_bits * repeat}, dapat {arr.size}"
        )

    hard = (arr[:n_payload_bits * repeat] >= 0.5).astype(np.int64)
    copies = hard.reshape(repeat, n_payload_bits)
    voted = (copies.sum(axis=0) > (repeat / 2)).astype(np.int64)
    return voted


def bits_to_text_ecc(bits, n_payload_bits=PAYLOAD_BITS, repeat=ECC_REPEAT):
    """Bit ECC -> majority-vote -> karakter ASCII printable."""
    voted = ecc_majority_vote(bits, n_payload_bits, repeat)
    chars = []
    for i in range(0, len(voted) - len(voted) % 8, 8):
        val = int(''.join(map(str, voted[i:i + 8])), 2)
        if 32 <= val <= 126:
            chars.append(chr(val))
    return ''.join(chars)


def ecc_soft_vote(probabilities, n_payload_bits=PAYLOAD_BITS, repeat=ECC_REPEAT):
    """Gabungkan probabilitas antarsalinan ECC sebelum threshold.

    Soft-vote mempertahankan confidence decoder dan lebih stabil daripada
    majority-vote setelah seluruh nilai lebih dulu dibulatkan.
    """
    arr = np.asarray(probabilities, dtype=np.float32).reshape(-1)
    expected = n_payload_bits * repeat
    if arr.size < expected:
        raise ValueError(f"Probabilitas ECC kurang: butuh {expected}, dapat {arr.size}")
    copies = arr[:expected].reshape(repeat, n_payload_bits)
    payload_probs = copies.mean(axis=0)
    return (payload_probs >= 0.5).astype(np.int64), payload_probs


def payload_bits_to_text(payload_bits):
    bits = np.asarray(payload_bits).astype(np.int64).reshape(-1)
    chars = []
    for i in range(0, len(bits) - len(bits) % 8, 8):
        value = int(''.join(map(str, bits[i:i + 8])), 2)
        chars.append(chr(value) if 32 <= value <= 126 else '�')
    return ''.join(chars)


def probabilities_to_text_ecc(probabilities):
    payload_bits, _ = ecc_soft_vote(probabilities)
    return payload_bits_to_text(payload_bits)


def make_random_ecc_watermark(batch_size=1):
    """Buat payload 48-bit acak lalu ulangi payload utuh ECC_REPEAT kali.

    Ini berguna untuk sanity test: strukturnya sama seperti payload produksi,
    tetapi nilainya BUKAN 'sabila'.
    """
    raw = torch.randint(
        0, 2, (batch_size, PAYLOAD_BITS), device=device
    ).float()
    encoded = raw.repeat(1, ECC_REPEAT)
    return encoded, raw


wm_bits_list_ecc, _payload_len = text_to_bits_ecc(
    WATERMARK_TEXT, ECC_REPEAT, WATERMARK_LENGTH
)
assert _payload_len == PAYLOAD_BITS
GROUND_TRUTH_WM = torch.tensor(
    [wm_bits_list_ecc], dtype=torch.float32, device=device
)

assert bits_to_text_ecc(
    GROUND_TRUTH_WM[0].detach().cpu().numpy()
) == WATERMARK_TEXT

print(f"Ground truth text : '{WATERMARK_TEXT}'")
print(f"Ground truth bits : {tuple(GROUND_TRUTH_WM.shape)}")
print(
    "Decode check      :",
    bits_to_text_ecc(GROUND_TRUTH_WM[0].detach().cpu().numpy())
)


## 3. Preprocessing Video (Ekstraksi Frame, Resize, Normalisasi)

Mengambil frame dari setiap video di Google Drive, melakukan resize ke `FRAME_SIZE`, dan normalisasi nilai piksel ke rentang [0, 1].

In [ ]:
import cv2
import subprocess
import tempfile
import matplotlib.pyplot as plt

def extract_frames(video_path, frame_size=FRAME_SIZE, max_frames=MAX_FRAMES_PER_VIDEO):
    """Ekstrak frame RGB float32 [0,1] beserta fps dan ukuran sumber."""
    video_path = str(video_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video tidak dapat dibuka: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 25.0
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (frame_size[1], frame_size[0]))
        frames.append(frame.astype(np.float32) / 255.0)
    cap.release()

    if not frames:
        raise RuntimeError(f"Tidak ada frame yang dapat diekstrak: {video_path}")
    return frames, fps, (orig_w, orig_h)


def frames_to_video(frames, out_path, fps, size=None):
    """Simpan RGB [0,1] ke video mp4v; hanya untuk preview, bukan sumber eksperimen."""
    if not frames:
        raise ValueError("frames kosong")
    if size is None:
        height, width = frames[0].shape[:2]
    else:
        width, height = size
    writer = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height)
    )
    if not writer.isOpened():
        raise RuntimeError(f"VideoWriter gagal dibuka: {out_path}")
    for frame in frames:
        frame_u8 = np.clip(frame * 255.0, 0, 255).astype(np.uint8)
        frame_u8 = cv2.resize(frame_u8, (width, height))
        writer.write(cv2.cvtColor(frame_u8, cv2.COLOR_RGB2BGR))
    writer.release()
    return str(out_path)


def frames_to_video_lossless(frames, out_path, fps, size=None):
    """Simpan intermediate sebagai FFV1 lossless agar codec uji adalah satu-satunya lossy stage."""
    if not frames:
        raise ValueError("frames kosong")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if size is None:
        height, width = frames[0].shape[:2]
    else:
        width, height = size

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        for i, frame in enumerate(frames):
            frame_u8 = np.clip(frame * 255.0, 0, 255).astype(np.uint8)
            frame_u8 = cv2.resize(frame_u8, (width, height))
            cv2.imwrite(
                str(tmpdir / f'f_{i:05d}.png'),
                cv2.cvtColor(frame_u8, cv2.COLOR_RGB2BGR),
            )
        command = [
            'ffmpeg', '-y', '-loglevel', 'error',
            '-framerate', str(fps), '-i', str(tmpdir / 'f_%05d.png'),
            '-an', '-c:v', 'ffv1', str(out_path),
        ]
        subprocess.run(command, check=True)
    return str(out_path)


def save_frames_as_images(frames, save_dir, prefix='frame', limit=12):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = []
    for i, frame in enumerate(frames[:limit]):
        out_path = save_dir / f'{prefix}_{i + 1:04d}.jpg'
        frame_u8 = np.clip(frame * 255.0, 0, 255).astype(np.uint8)
        cv2.imwrite(str(out_path), cv2.cvtColor(frame_u8, cv2.COLOR_RGB2BGR))
        saved_paths.append(out_path)
    return saved_paths


def show_frame_grid(frames, n_show=12, title='Preview frame'):
    n_show = min(n_show, len(frames))
    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.asarray(axes).reshape(-1)
    indices = np.linspace(0, len(frames) - 1, n_show, dtype=int)
    for axis, index in zip(axes, indices):
        axis.imshow(frames[index])
        axis.set_title(f'Frame {index + 1}')
        axis.axis('off')
    for axis in axes[len(indices):]:
        axis.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


dataset_frames = {}
for video_name in video_list:
    frames, fps, original_size = extract_frames(VIDEO_PATHS[video_name])
    dataset_frames[video_name] = (frames, fps, original_size)
    split_name = 'train' if video_name in set(TRAIN_VIDEO_NAMES) else 'evaluation'
    print(
        f"[{split_name:10}] {video_name}: {len(frames)} frame, "
        f"fps={fps:.2f}, ukuran={original_size}"
    )

preview_name = EVAL_VIDEO_NAMES[0]
preview_frames = dataset_frames[preview_name][0]
preview_dir = FRAMES_DIR / Path(preview_name).stem
preview_paths = save_frames_as_images(preview_frames, preview_dir, limit=12)
show_frame_grid(preview_frames, n_show=12, title=f'Preview evaluation: {preview_name}')
print(f"Preview disimpan: {len(preview_paths)} gambar di {preview_dir}")


## 4. Neural Watermark Encoder & Decoder (Arsitektur Model)

Model watermarking berbasis CNN dengan pendekatan *encoder-decoder* (mirip arsitektur HiDDeN):

- **Encoder**: menerima frame asli + bit watermark, menghasilkan *residual* halus yang ditambahkan ke frame sehingga watermark tidak terlihat kasat mata (*invisible watermark*).
- **Decoder**: menerima frame (bisa sudah terkompresi/terdistorsi) dan berusaha mengekstrak kembali bit watermark.
- **Noise Layer**: disisipkan di antara encoder dan decoder **saat training** untuk mensimulasikan distorsi kompresi (blur, noise, JPEG-like quantization), agar model belajar menghasilkan watermark yang **tahan kompresi tinggi**.


**Catatan:** kekuatan residual watermark dinaikkan sedikit (0.12 -> 0.15) karena training sekarang juga harus tahan terhadap kompresi Neural Codec yang nyata (bukan hanya noise sintetis), yang lebih destruktif.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import random

class WatermarkEncoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=64):
        super().__init__()
        self.wm_length = wm_length
        # Proyeksikan bit watermark ke peta kecil (8x8) dulu, BUKAN langsung ke 128x128
        self.fc_wm = nn.Linear(wm_length, 16 * 8 * 8)
        self.wm_upsample = nn.Sequential(
            nn.ConvTranspose2d(16, 32, 4, stride=2, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=True),  # 8->16
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.GroupNorm(4, 16), nn.ReLU(inplace=True),  # 16->32
            nn.ConvTranspose2d(16, 8, 4, stride=2, padding=1), nn.GroupNorm(2, 8), nn.ReLU(inplace=True),    # 32->64
            nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1), nn.GroupNorm(2, 4), nn.ReLU(inplace=True),     # 64->128
        )
        self.conv_in = nn.Sequential(
            nn.Conv2d(3 + 4, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_mid = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_out = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, frame, wm_bits):
        B, C, H, W = frame.shape
        wm_feat = self.fc_wm(wm_bits).view(B, 16, 8, 8)
        wm_map = self.wm_upsample(wm_feat)          # (B, 4, 128, 128)
        x = torch.cat([frame, wm_map], dim=1)
        x = self.conv_in(x)
        x = self.conv_mid(x) + x
        residual = torch.tanh(self.conv_out(x)) * 0.10  # dinaikkan lagi 0.08 -> 0.10: training
        # sekarang menghadapi ffmpeg H.264/H.265 SUNGGUHAN (Bagian 5, ffmpeg_bpda_compress) yang
        # jauh lebih destruktif daripada Neural Codec buatan sendiri atau noise sintetis.
        # LAMBDA_BIAS (anti color-cast) tetap menjaga agar kenaikan ini tidak muncul sebagai
        # blok warna kasat mata, melainkan tekstur noise halus yang tersebar.
        watermarked = torch.clamp(frame + residual, 0.0, 1.0)
        return watermarked, residual


class WatermarkDecoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Sequential(
            nn.Linear(channels * 4 * 4, 128), nn.ReLU(inplace=True),
            nn.Linear(128, wm_length),
        )

    def forward(self, frame):
        x = self.features(frame)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

print("Arsitektur dengan GroupNorm berhasil didefinisikan.")

## [FIX #3] Encoder/Decoder V2 — Kapasitas & Kekuatan Residual Dinaikkan

Payload sekarang 144 bit (naik dari 64 bit) setelah ECC — anggaran sinyal & kapasitas versi
lama terlalu ketat untuk payload sebesar ini ditambah *noise curriculum* yang berat. Sel ini
MENGGANTIKAN (bukan menghapus) definisi kelas `WatermarkEncoder`/`WatermarkDecoder` dari
Bagian 4 di atas untuk seluruh sel setelah ini — sel Bagian 4 sendiri tidak diubah.

- `channels`: 64 → 80
- Ditambah satu blok residual (`conv_mid2`) pada encoder
- Kekuatan residual watermark: 0.10 → **0.18**


In [ ]:
print("=== [FIX #3] Encoder/Decoder V2: kapasitas & kekuatan residual dinaikkan ===")
print("channels 64->80, +1 blok residual, kekuatan residual watermark 0.10->0.18")
print("(payload sekarang 144 bit setelah ECC, anggaran sinyal versi lama terlalu ketat).")


class WatermarkEncoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.wm_length = wm_length
        self.fc_wm = nn.Linear(wm_length, 16 * 8 * 8)
        self.wm_upsample = nn.Sequential(
            nn.ConvTranspose2d(16, 32, 4, stride=2, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.GroupNorm(4, 16), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 8, 4, stride=2, padding=1), nn.GroupNorm(2, 8), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1), nn.GroupNorm(2, 4), nn.ReLU(inplace=True),
        )
        self.conv_in = nn.Sequential(
            nn.Conv2d(3 + 4, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_mid = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        # [FIX #3] blok residual tambahan -- kapasitas lebih besar untuk payload 144-bit.
        self.conv_mid2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_out = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, frame, wm_bits):
        B, C, H, W = frame.shape
        wm_feat = self.fc_wm(wm_bits).view(B, 16, 8, 8)
        wm_map = self.wm_upsample(wm_feat)
        x = torch.cat([frame, wm_map], dim=1)
        x = self.conv_in(x)
        x = self.conv_mid(x) + x
        x = self.conv_mid2(x) + x
        # [FIX #3] kekuatan residual dinaikkan dari 0.10 -> 0.18.
        residual = torch.tanh(self.conv_out(x)) * 0.18
        watermarked = torch.clamp(frame + residual, 0.0, 1.0)
        return watermarked, residual


class WatermarkDecoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Sequential(
            nn.Linear(channels * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Linear(256, wm_length),
        )

    def forward(self, frame):
        x = self.features(frame)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)


print("Arsitektur V2 di atas MENGGANTIKAN definisi Bagian 4 untuk sel-sel berikutnya "
      "(sel Bagian 4 sendiri TIDAK diubah/dihapus).")


## 4B. Neural Codec — Compressive Autoencoder (Dilatih Lebih Dahulu, dengan Gradien Diperbaiki)

Di versi sebelumnya, Neural Codec baru dilatih **setelah** encoder-decoder watermark selesai — bahkan gagal jalan karena variabel `all_frames_t`/`n` tidak pernah didefinisikan (`NameError`). Akibatnya Neural Codec **tidak pernah ikut serta** dalam training watermark, padahal judul penelitian ini eksplisit menyebut **Neural Codec**.

Perbaikan pada bagian ini:

1. **Urutan dipindah ke awal** — Neural Codec dilatih di sini, sebelum training watermark (Bagian 5), sehingga bisa dipakai sebagai *noise layer* nyata saat watermark dilatih.
2. **Bug variabel tidak terdefinisi diperbaiki** — frame training diambil langsung dari `dataset_frames` yang sudah ada.
3. **Straight-Through Estimator (STE) untuk kuantisasi** — sebelumnya `torch.round()` memutus gradien ke bagian *encoder* Neural Codec sehingga bagian itu tidak pernah ter-update (hanya *decoder*-nya yang belajar). Dengan STE, forward pass tetap terkuantisasi (tetap lossy & realistis), tapi gradien tetap mengalir saat backward pass.
4. Setelah dilatih, bobot Neural Codec **dibekukan** (`requires_grad=False`) lalu dipakai sebagai salah satu jenis distorsi asli di dalam training loop watermark (Bagian 5) — bukan sekadar noise sintetis pengganti.

In [ ]:
def ste_quantize(z, levels=16):
    """Straight-Through Estimator: forward pakai versi terkuantisasi (realistis & lossy),
    backward memperlakukan kuantisasi seolah identity, supaya gradien tetap mengalir
    ke layer sebelumnya (Encoder Neural Codec)."""
    z_hard = torch.round(z * levels) / levels
    return z + (z_hard - z).detach()


class NeuralCodec(nn.Module):
    """Autoencoder ringan dengan bottleneck sempit + kuantisasi (STE),
    mensimulasikan kompresi berbasis neural network pada level frame."""
    def __init__(self, bottleneck=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),   # /2
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),  # /4
            nn.Conv2d(64, bottleneck, 3, padding=1),                          # bottleneck sempit
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(bottleneck, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 3, padding=1),
        )

    def forward(self, x, quant_levels=16):
        z = torch.tanh(self.enc(x))
        z = ste_quantize(z, quant_levels)      # kuantisasi lossy, gradien TETAP mengalir (STE)
        out = torch.sigmoid(self.dec(z))
        return out


neural_codec = NeuralCodec().to(device)

# --- Siapkan data training Neural Codec (bug lama: all_frames_t & n tidak didefinisikan) ---
train_video_names = [name for name in TRAIN_VIDEO_NAMES if name in dataset_frames]
eval_video_names = [name for name in EVAL_VIDEO_NAMES if name in dataset_frames]
assert set(train_video_names).isdisjoint(eval_video_names)
print(f"Video training: {len(train_video_names)} | evaluation hold-out: {len(eval_video_names)}")

CODEC_TRAIN_FRAMES = 300   # batasi jumlah frame supaya hemat memori GPU di Colab
CODEC_BATCH_SIZE = 8

codec_frame_pool = []
for vname in train_video_names:
    frames, _, _ = dataset_frames[vname]
    codec_frame_pool.extend(frames)
    if len(codec_frame_pool) >= CODEC_TRAIN_FRAMES:
        break
codec_frame_pool = codec_frame_pool[:CODEC_TRAIN_FRAMES]

all_frames_t = torch.tensor(np.stack(codec_frame_pool)).permute(0, 3, 1, 2).float().to(device)
n = all_frames_t.size(0)
print(f"Neural Codec akan dilatih dengan {n} frame.")

# --- Latih Neural Codec (self-supervised: merekonstruksi frame aslinya sendiri) ---
codec_opt = torch.optim.Adam(neural_codec.parameters(), lr=1e-3)
NEURAL_CODEC_EPOCHS = 60

for epoch in range(NEURAL_CODEC_EPOCHS):
    perm = torch.randperm(n)
    ep_loss = 0.0
    for i in range(0, n, CODEC_BATCH_SIZE):
        idx = perm[i:i + CODEC_BATCH_SIZE]
        batch = all_frames_t[idx]
        recon = neural_codec(batch)
        loss = F.mse_loss(recon, batch)
        codec_opt.zero_grad()
        loss.backward()
        codec_opt.step()
        ep_loss += loss.item() * batch.size(0)
    print(f"[Neural Codec] Epoch {epoch+1}/{NEURAL_CODEC_EPOCHS} | recon_loss={ep_loss/n:.5f}")

# Bekukan Neural Codec: dipakai sebagai noise layer TETAP saat training watermark
# (parameternya tidak lagi dioptimasi, tapi gradien tetap bisa lewat lewat input-nya).
for p in neural_codec.parameters():
    p.requires_grad_(False)
neural_codec.eval()
print("Neural Codec selesai dilatih & dibekukan — siap dipakai sebagai noise layer nyata di training watermark (Bagian 5).")

## 5. Training Neural Watermark Encoder-Decoder

Training dilakukan singkat (beberapa epoch) menggunakan frame-frame dari dataset di Google Drive, dengan bit watermark acak sebagai label. Noise layer disisipkan agar model tahan kompresi.

**Update:** noise layer saat training sekarang mencakup kompresi **Neural Codec nyata** (hasil Bagian 4B, bukan proxy sintetis) selain gaussian/blur/quantize, ditambah augmentasi flip horizontal acak untuk membantu keterbatasan jumlah video dataset.

In [ ]:
import gc, random
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("CATATAN: mulai epoch ini, noise layer sesekali memanggil ffmpeg ASLI (h264_real/h265_real) "
      "lewat BPDA -> training akan LEBIH LAMBAT dari sebelumnya (tiap batch yang kena mode ini "
      "menulis PNG + encode ffmpeg beneran). Kalau di Colab free tier terlalu lambat/timeout, "
      "kurangi MAX_VIDEOS_FOR_TRAINING atau EPOCHS, JANGAN kurangi bobot h264_real/h265_real di "
      "apply_curriculum_noise -- itu justru bagian yang menutup domain gap.")

encoder = WatermarkEncoder().to(device)
decoder = WatermarkDecoder().to(device)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

EPOCHS = 160  # dinaikkan dari 80 -> 160 (2x). Kalau setelah ini masih jauh dari
# target 0.95, TIDAK PERLU ulang dari awal -- cukup naikkan angka ini lagi lalu jalankan
# ulang sel ini; training otomatis lanjut dari checkpoint terakhir (lihat CHECKPOINT_PATH
# di bawah), bukan dari epoch 1.
BATCH_SIZE = 8
WARMUP_EPOCHS = 5          # arsitektur sudah sehat, belajar cepat
TARGET_BIT_ACC = 0.95      # target akhir (gabungan semua noise); dipakai utk gate LAMBDA saja
TARGET_BIT_ACC_REAL = 0.92  # BARU: target akurasi KHUSUS di bawah h264_real/h265_real --
# inilah yang sekarang menentukan kapan training berhenti (bukan bit_acc gabungan di atas,
# yang bisa "curang" naik gara-gara noise sintetis lebih gampang). 0.92 dipilih sebagai
# target awal menuju 90-an; naikkan lagi (mis. 0.95) kalau mau lebih ketat setelah ini tercapai.

# PERBAIKAN BUG CURRICULUM CRF (root cause bit hasil ekstraksi acak):
# Sebelumnya, kecepatan curriculum CRF (di apply_curriculum_noise) dihitung relatif
# terhadap EPOCHS -- setiap kali EPOCHS dinaikkan (80->160->400...) untuk melanjutkan
# training, curriculum jadi makin LAMBAT mencapai CRF penuh pada epoch yang sama.
# Akibatnya model bisa "lulus" TARGET_BIT_ACC_REAL padahal CRF training yang benar-benar
# dilihat baru ~20-30, belum pernah menyentuh CRF_HIGH_COMPRESSION=35 yang dipakai saat
# pengujian nyata (Bagian 7a) -- itulah sebab bit hasil ekstraksi acak/tidak sesuai teks.
# CRF_CURRICULUM_EPOCHS di bawah ini adalah PATOKAN TETAP (jumlah epoch sejak warmup
# sampai curriculum CRF mencapai penuh/CRF~35-38) -- TIDAK ikut melar lagi walau EPOCHS
# dinaikkan lagi nanti untuk melanjutkan training.
CRF_CURRICULUM_EPOCHS = 100

# train_video_names sudah didefinisikan di Bagian 4B, dipakai ulang di sini
print(f"Training menggunakan {len(train_video_names)} video. Warm-up: {WARMUP_EPOCHS} epoch pertama.")


def augment_batch(x):
    """Augmentasi flip horizontal acak PER-BATCH (bukan lagi per-sampel individual).
    Sejak batch = potongan klip video berurutan (dipakai juga sbg input ffmpeg BPDA),
    flip per-frame independen akan membuat klip 'flicker' tidak natural (separuh frame
    terbalik, separuh tidak) -- yang lagi-lagi membuat kompresi H.264/H.265 nyata
    menghasilkan artefak yang tidak representatif."""
    if torch.rand(1).item() < 0.5:
        x = torch.flip(x, dims=[3])
    return x


def ffmpeg_bpda_compress(x, fps=25, crf_range=(25, 38), codecs=('libx264',), preset='fast'):
    """PERBAIKAN UTAMA (menutup domain gap): kompresi H.264/H.265 SUNGGUHAN lewat
    ffmpeg dipakai langsung sebagai noise layer saat training -- bukan lagi proxy
    sintetis (gaussian/blur/quantize) yang artefaknya sama sekali berbeda dari
    kompresi berbasis blok DCT + motion compensation + deblocking filter.

    ffmpeg tidak differentiable, jadi dipakai pendekatan BPDA (Backward Pass
    Differentiable Approximation), sama triknya dengan STE yang sudah dipakai
    untuk NeuralCodec: forward pass memakai hasil kompresi ffmpeg NYATA (lossy,
    realistis), backward pass memperlakukan operasi ini seolah identity supaya
    gradien tetap mengalir ke encoder.

    crf_range dibuat sedikit LEBIH LEBAR dari crf pengujian (CRF_HIGH_COMPRESSION=35)
    supaya model juga tahan di sekitar titik itu, bukan hanya pas di satu nilai.
    """
    B, C, H, W = x.shape
    codec = random.choice(codecs)
    crf = random.randint(*crf_range)

    with tempfile.TemporaryDirectory() as tmpdir:
        frames_np = x.detach().permute(0, 2, 3, 1).cpu().numpy()
        png_dir = os.path.join(tmpdir, 'png')
        os.makedirs(png_dir, exist_ok=True)
        for i, f in enumerate(frames_np):
            f_uint8 = np.clip(f * 255.0, 0, 255).astype(np.uint8)
            cv2.imwrite(os.path.join(png_dir, f'f_{i:04d}.png'), cv2.cvtColor(f_uint8, cv2.COLOR_RGB2BGR))

        out_path = os.path.join(tmpdir, 'out.mp4')
        cmd = [
            'ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(png_dir, 'f_%04d.png'),
            '-c:v', codec, '-crf', str(crf), '-preset', preset, '-pix_fmt', 'yuv420p',
            out_path
        ]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        cap = cv2.VideoCapture(out_path)
        out_frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (W, H))
            out_frames.append(frame.astype(np.float32) / 255.0)
        cap.release()

        # Jaga-jaga bila ffmpeg drop/duplikasi frame (kadang terjadi pada klip sangat pendek)
        while len(out_frames) < B:
            out_frames.append(out_frames[-1] if out_frames else frames_np[len(out_frames)])
        out_frames = out_frames[:B]

    x_compressed = torch.tensor(np.stack(out_frames)).permute(0, 3, 1, 2).float().to(x.device)
    return x + (x_compressed - x).detach()   # BPDA: forward=ffmpeg asli, backward=identity


def apply_curriculum_noise(x, epoch, total_epochs, prev_bit_acc=0.5):
    """Noise layer saat training.
    - 'neural' memakai Neural Codec asli (Bagian 4B) yang sudah dilatih & dibekukan.
    - 'h264_real'/'h265_real' memakai ffmpeg H.264/H.265 SUNGGUHAN lewat BPDA -- inilah
      kompresi yang benar-benar dipakai saat pengujian (Bagian 7a).
    - PERBAIKAN (curriculum berbasis performa): noise real ini juga menunggu decoder
      sudah punya sedikit pegangan (prev_bit_acc >= 0.55, bukan cuma progress waktu),
      supaya tidak dihajar noise terberat saat decoder masih tebak-acak total. Ada
      SAFETY VALVE: dipaksa aktif setelah WARMUP_EPOCHS+40 epoch apa pun kondisinya,
      supaya tidak tertunda selamanya kalau bit_acc tidak kunjung naik.
    """
    if epoch <= WARMUP_EPOCHS:
        return x, 'none'
    # PERBAIKAN: progress SEKARANG dipatok ke CRF_CURRICULUM_EPOCHS (konstanta TETAP),
    # bukan lagi ke total_epochs/EPOCHS (yang berubah-ubah tiap kali dinaikkan untuk
    # melanjutkan training). Ini mencegah curriculum CRF melambat lagi di masa depan.
    progress = min((epoch - WARMUP_EPOCHS) / CRF_CURRICULUM_EPOCHS, 1.0)

    pool = ['gaussian', 'blur', 'quantize', 'neural']
    real_noise_ready = progress > 0.15 and (
        prev_bit_acc >= 0.55 or epoch > WARMUP_EPOCHS + 40
    )
    if real_noise_ready:
        pool += ['h264_real', 'h265_real']
    mode = random.choice(pool)

    if mode == 'gaussian':
        x = x + torch.randn_like(x) * (0.005 + 0.02 * progress)
    elif mode == 'blur' and progress > 0.3:
        x = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
    elif mode == 'quantize':
        # Dikalibrasi lebih berat (96->64 sebelumnya, sekarang 64->28) supaya mendekati
        # keparahan distorsi CRF 35 asli, bukan sekadar kuantisasi ringan.
        levels = int(64 - 36 * progress)
        x = torch.round(x * levels) / levels
    elif mode == 'neural':
        x = neural_codec(x)   # kompresi Neural Codec NYATA, gradien tetap mengalir ke encoder
    elif mode in ('h264_real', 'h265_real'):
        codec = 'libx264' if mode == 'h264_real' else 'libx265'
        # PERBAIKAN: crf_range sekarang BENAR-BENAR mengikuti curriculum (naik bertahap),
        # bukan langsung lompat ke rentang lebar (25-38) sejak mode ini pertama aktif.
        # sub_progress: 0 saat baru diaktifkan (progress=0.15) -> 1 di akhir training.
        sub_progress = min(max((progress - 0.15) / 0.85, 0.0), 1.0)
        crf_lo = int(18 + 10 * sub_progress)   # 18 -> 28
        crf_hi = int(28 + 10 * sub_progress)   # 28 -> 38 (mencakup crf uji=35 di rentang akhir)
        # preset='fast' disamakan PERSIS dengan compress_ffmpeg() di Bagian 7a (evaluasi),
        # supaya karakter artefak yang dipelajari cocok dengan yang diuji.
        x = ffmpeg_bpda_compress(x, crf_range=(crf_lo, crf_hi), codecs=(codec,), preset='fast')
    # PERBAIKAN: kembalikan mode juga -- training loop butuh ini untuk melacak akurasi
    # KHUSUS real-codec (h264_real/h265_real) terpisah dari rata-rata gabungan semua noise,
    # supaya "bit_acc" yang dilaporkan tidak tercampur/terdongkrak oleh noise yang lebih mudah.
    return torch.clamp(x, 0.0, 1.0), mode


REBIT_EVERY_N_BATCHES = 1    # PHASE-1 FIX: payload BARU setiap batch/clip
# PHASE-1 FIX: setiap batch video memakai SATU payload acak yang sama untuk semua frame.
# Batch berikutnya mendapat payload baru. Ini menjaga watermark temporal tetap konsisten
# untuk H.264/H.265 sekaligus mencegah decoder menghafal satu target tetap.
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint_general_payload_v5_group_split.pth')
CHECKPOINT_EVERY = 10  # simpan checkpoint tiap 10 epoch (selain otomatis tiap kali cell berhenti/selesai)

global_step = 0
current_wm_bits_single = torch.randint(0, 2, (1, WATERMARK_LENGTH)).float().to(device)
history = {'loss': [], 'bit_acc': []}
start_epoch = 1

# RESUME: kalau ada checkpoint dari run sebelumnya (mis. sesi Colab putus di tengah, atau
# kamu sengaja menaikkan EPOCHS untuk lanjut training), lanjutkan dari situ -- bukan dari
# epoch 1. Kalau belum ada checkpoint, training mulai dari awal seperti biasa.
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    encoder.load_state_dict(ckpt['encoder'])
    decoder.load_state_dict(ckpt['decoder'])
    optimizer.load_state_dict(ckpt['optimizer'])
    history = ckpt['history']
    global_step = ckpt['global_step']
    current_wm_bits_single = ckpt.get(
        'current_wm_bits_single',
        torch.randint(0, 2, (1, WATERMARK_LENGTH)).float()
    ).to(device)
    start_epoch = ckpt['epoch'] + 1
    print(f"[RESUME] Checkpoint ditemukan -> melanjutkan dari epoch {start_epoch} "
          f"(sudah {len(history['loss'])} epoch tersimpan, bit_acc terakhir="
          f"{history['bit_acc'][-1]:.3f}).")
    if start_epoch > EPOCHS:
        print(f"[RESUME] EPOCHS ({EPOCHS}) sudah tercapai/terlewati oleh checkpoint. "
              f"Naikkan EPOCHS kalau mau training lebih lanjut, lalu jalankan ulang sel ini.")
else:
    print("[RESUME] Tidak ada checkpoint -- training dimulai dari epoch 1.")

# prev_bit_acc: dipakai untuk gerbang LAMBDA_IMG/BIAS & aktivasi noise real (lihat di atas).
# Kalau resume dari checkpoint, lanjutkan dari bit_acc epoch terakhir yang tersimpan --
# bukan dianggap "acak lagi" dari nol.
prev_bit_acc = history['bit_acc'][-1] if history['bit_acc'] else 0.5


def save_checkpoint(epoch):
    torch.save({
        'encoder': encoder.state_dict(),
        'decoder': decoder.state_dict(),
        'optimizer': optimizer.state_dict(),
        'history': history,
        'global_step': global_step,
        'current_wm_bits_single': current_wm_bits_single.cpu(),
        'epoch': epoch,
    }, CHECKPOINT_PATH)


for epoch in range(start_epoch, EPOCHS + 1):
    if epoch <= WARMUP_EPOCHS:
        LAMBDA_WM, LAMBDA_IMG, LAMBDA_BIAS, gate = 1.0, 0.0, 0.0, 0.0
    else:
        ramp = min((epoch - WARMUP_EPOCHS) / 8, 1.0)
        # PERBAIKAN: LAMBDA_IMG/BIAS sekarang DIGERBANG oleh bit_acc epoch sebelumnya
        # (bukan cuma waktu/ramp). Sebelumnya ramp selesai penuh di epoch WARMUP+8=13,
        # padahal noise real (h264_real/h265_real) baru mulai muncul >epoch 16 -- persis
        # saat itu tekanan "mirip gambar asli" sudah di puncak selama beberapa epoch,
        # sehingga encoder tidak diberi ruang menaikkan kekuatan sinyal untuk melawan
        # noise baru itu (tarik-menarik dua loss yang tidak sinkron).
        # gate: 0.1 (floor, tidak pernah nol total) saat prev_bit_acc <= 0.55 (~acak),
        # naik linear ke 1.0 saat prev_bit_acc >= 0.85. Floor 0.1 sengaja dipasang supaya
        # tekanan invisibility tidak hilang SAMA SEKALI kalau bit_acc lama tersendat --
        # tanpa floor ini, residual bisa dibiarkan maksimal terus-menerus tanpa insentif
        # mengecil sama sekali, yang berisiko merusak metrik PSNR/SSIM akhir.
        gate = max(0.1, min((prev_bit_acc - 0.55) / (0.85 - 0.55), 1.0)) if prev_bit_acc > 0.55 else 0.1
        LAMBDA_WM, LAMBDA_IMG, LAMBDA_BIAS = 1.0, 1.0 * ramp * gate, 5.0 * ramp * gate
        # LAMBDA_BIAS menghukum PERGESERAN WARNA RATA (color-cast) -- penyebab utama
        # tampilan hijau/pink menutup seluruh frame. Ini memaksa residual menjadi tekstur
        # bernoise halus yang tersebar (kurang kasat mata), bukan blok warna rata.

    epoch_loss, epoch_correct_bits, epoch_total_bits, total_samples = 0.0, 0, 0, 0
    # PERBAIKAN: hitung akurasi KHUSUS batch yang kena noise real (h264_real/h265_real)
    # secara terpisah dari rata-rata gabungan semua noise -- supaya tidak tertutupi oleh
    # noise sintetis yang jauh lebih mudah dilewati.
    epoch_correct_bits_real, epoch_total_bits_real = 0, 0

    for vname in train_video_names:
        frames, fps, size = dataset_frames[vname]
        frames_np = np.stack(frames)
        n_local = frames_np.shape[0]

        # PERBAIKAN BUG PENTING: sebelumnya batch diambil dari permutasi ACAK seluruh
        # frame video (perm_local). Untuk noise sintetis per-frame (gaussian/blur/quantize)
        # itu tidak masalah -- tapi begitu batch acak ini dipakai sebagai "video" untuk
        # ffmpeg_bpda_compress (h264_real/h265_real), 8 frame yang melompat acak dalam waktu
        # itu di-encode H.264/H.265 SUNGGUHAN dengan motion compensation antar-frame yang
        # mencari korelasi gerak yang TIDAK ADA (karena bukan potongan video asli) --
        # menghasilkan artefak yang jauh lebih destruktif & tidak representatif dibanding
        # CRF 35 pada video sungguhan berurutan yang dipakai saat pengujian (Bagian 7a).
        # Sekarang: pilih sejumlah titik mulai (diacak urutannya per epoch), lalu ambil
        # BATCH_SIZE frame BERURUTAN (sliding window) per batch -- variasi tetap didapat
        # dari titik mulai yang acak & video yang berganti-ganti, tapi urutan temporal DI
        # DALAM satu batch tetap valid seperti video sungguhan.
        starts = list(range(0, max(n_local - BATCH_SIZE, 0) + 1, BATCH_SIZE))
        if not starts:
            starts = [0]
        random.shuffle(starts)

        for start in starts:
            idx_local = np.arange(start, min(start + BATCH_SIZE, n_local))
            batch = torch.tensor(frames_np[idx_local]).permute(0, 3, 1, 2).float().to(device)
            batch = augment_batch(batch)
            B = batch.size(0)

            # PHASE-1 FIX: satu payload acak konsisten sepanjang clip; ganti tiap batch.
            if global_step % REBIT_EVERY_N_BATCHES == 0:
                current_wm_bits_single = torch.randint(
                    0, 2, (1, WATERMARK_LENGTH), device=device
                ).float()

            # Satu payload yang SAMA untuk semua frame dalam clip/batch.
            # Batch berikutnya mendapat payload acak baru.
            wm_bits = current_wm_bits_single.repeat(B, 1)
            global_step += 1

            watermarked, residual = encoder(batch, wm_bits)
            noised, noise_mode = apply_curriculum_noise(watermarked, epoch, EPOCHS, prev_bit_acc=prev_bit_acc)
            pred_logits = decoder(noised)

            loss_img = mse_loss(watermarked, batch)
            loss_wm = bce_loss(pred_logits, wm_bits)
            # Penalti anti color-cast: rata-ratakan residual per channel per sampel (buang
            # dimensi spasial) lalu kuadratkan. Kalau residual punya pergeseran warna rata
            # (bukan tekstur seimbang), nilai ini besar -> dihukum.
            loss_bias = residual.mean(dim=(2, 3)).pow(2).mean()
            loss = LAMBDA_IMG * loss_img + LAMBDA_WM * loss_wm + LAMBDA_BIAS * loss_bias

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                pred_bits = (torch.sigmoid(pred_logits) > 0.5).float()
                n_correct = (pred_bits == wm_bits).sum().item()
                epoch_correct_bits += n_correct
                epoch_total_bits += wm_bits.numel()
                if noise_mode in ('h264_real', 'h265_real'):
                    epoch_correct_bits_real += n_correct
                    epoch_total_bits_real += wm_bits.numel()

            epoch_loss += loss.item() * B
            total_samples += B
            del batch, watermarked, noised, pred_logits

        del frames, frames_np

    gc.collect(); torch.cuda.empty_cache()

    bit_acc = epoch_correct_bits / epoch_total_bits
    # PERBAIKAN: bit_acc_real = akurasi KHUSUS di bawah h264_real/h265_real -- ini yang
    # sebenarnya relevan untuk klaim skripsi ("tahan kompresi tinggi"), bukan bit_acc
    # gabungan yang bisa didongkrak oleh noise sintetis yang lebih mudah. Kalau belum ada
    # batch yang kena noise real di epoch ini (mis. masih warm-up), pakai nilai epoch
    # sebelumnya supaya tidak melompat ke 0.
    if epoch_total_bits_real > 0:
        bit_acc_real = epoch_correct_bits_real / epoch_total_bits_real
    else:
        bit_acc_real = history.get('bit_acc_real', [0.0])[-1] if history.get('bit_acc_real') else 0.0
    avg_loss = epoch_loss / total_samples
    history['loss'].append(avg_loss)
    history['bit_acc'].append(bit_acc)
    history.setdefault('bit_acc_real', []).append(bit_acc_real)

    tag = "[WARM-UP]" if epoch <= WARMUP_EPOCHS else ""
    print(f"Epoch {epoch:02d}/{EPOCHS} {tag} | loss={avg_loss:.5f} | bit_acc={bit_acc:.4f} "
          f"| bit_acc_REAL(h264/h265)={bit_acc_real:.4f} | gate={gate:.2f} | LAMBDA_IMG={LAMBDA_IMG:.3f}")

    # Update prev_bit_acc UNTUK EPOCH BERIKUTNYA (gerbang & noise real memakai nilai epoch
    # SEBELUMNYA -- bukan bit_acc epoch berjalan -- supaya keputusan curriculum konsisten
    # sepanjang satu epoch, tidak berubah di tengah jalan).
    prev_bit_acc = bit_acc

    if epoch % CHECKPOINT_EVERY == 0:
        save_checkpoint(epoch)
        print(f"   [checkpoint tersimpan @ epoch {epoch}]")

    # PERBAIKAN: early-stop sekarang butuh bit_acc_REAL (bukan bit_acc gabungan) mencapai
    # target, DAN sudah ada cukup banyak batch real yang teramati epoch ini (>=20) supaya
    # keputusan berhenti tidak didasarkan pada sampel yang terlalu sedikit/kebetulan.
    # PERBAIKAN: early-stop sekarang JUGA mensyaratkan curriculum CRF sudah benar-benar
    # mencapai penuh (CRF training sudah mencakup ~35, sama dengan CRF_HIGH_COMPRESSION
    # yang dipakai saat pengujian nyata) -- bukan cuma target bit_acc_REAL tercapai di
    # CRF yang masih ringan seperti bug sebelumnya.
    curriculum_full_crf = (epoch - WARMUP_EPOCHS) >= CRF_CURRICULUM_EPOCHS
    target_reached = (bit_acc_real >= TARGET_BIT_ACC_REAL and epoch_total_bits_real >= 20 * WATERMARK_LENGTH
            and epoch >= WARMUP_EPOCHS + 5)
    if target_reached and curriculum_full_crf:
        print(f"\nTarget akurasi REAL-CODEC {TARGET_BIT_ACC_REAL:.0%} tercapai pada epoch {epoch} "
              f"(bit_acc gabungan={bit_acc:.4f}), DAN curriculum CRF sudah penuh sejak epoch "
              f"{WARMUP_EPOCHS + CRF_CURRICULUM_EPOCHS} (CRF training sudah mencakup ~35). "
              f"Training dihentikan lebih awal.")
        save_checkpoint(epoch)
        break
    elif target_reached and not curriculum_full_crf:
        print(f"   [info] bit_acc_REAL sudah capai target {TARGET_BIT_ACC_REAL:.0%}, TAPI curriculum "
              f"CRF belum penuh (epoch {epoch - WARMUP_EPOCHS}/{CRF_CURRICULUM_EPOCHS} sejak warmup) "
              f"-- training TIDAK dihentikan dulu supaya model benar-benar teruji di CRF~35.")
else:
    # loop selesai tanpa mencapai target -> tetap simpan checkpoint terakhir supaya bisa
    # dilanjutkan nanti dengan menaikkan EPOCHS, tanpa mengulang dari awal
    if EPOCHS >= start_epoch:
        save_checkpoint(EPOCHS)

torch.save(encoder.state_dict(), os.path.join(OUTPUT_DIR, 'encoder.pth'))
torch.save(decoder.state_dict(), os.path.join(OUTPUT_DIR, 'decoder.pth'))
print("Model tersimpan.")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(history['loss']); plt.title('Loss')
plt.subplot(1, 2, 2)
plt.plot(history['bit_acc'], label='bit_acc (gabungan semua noise)')
# PERBAIKAN: tambahkan kurva bit_acc_REAL supaya` perbedaan gabungan vs khusus h264/h265
# terlihat langsung -- ini angka yang sebenarnya relevan untuk klaim "tahan kompresi tinggi".
if 'bit_acc_real' in history:
    plt.plot(history['bit_acc_real'], label='bit_acc REAL (h264/h265 saja)')
plt.axhline(TARGET_BIT_ACC, color='r', linestyle='--', label=f'target gabungan ({TARGET_BIT_ACC:.0%})')
plt.axhline(TARGET_BIT_ACC_REAL, color='g', linestyle='--', label=f'target REAL ({TARGET_BIT_ACC_REAL:.0%})')
plt.legend(fontsize=8)
plt.title('Bit Accuracy'); plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve.png'), dpi=150); plt.show()

print(f"\nAkurasi gabungan awal: {history['bit_acc'][0]:.2f} -> akhir: {history['bit_acc'][-1]:.2f}")
if history.get('bit_acc_real'):
    print(f"Akurasi REAL (h264/h265) awal: {history['bit_acc_real'][0]:.2f} -> akhir: {history['bit_acc_real'][-1]:.2f}")



## [DINONAKTIFKAN] Fine-tune V2 Target Tetap

Versi lama fine-tune ini menggunakan **`"sabila"` pada setiap batch**. Pola tersebut dapat
membuat decoder memperoleh akurasi 100% dengan mempelajari prior output, bukan membaca
residual watermark dari gambar.

Cell berikut dipertahankan sebagai **guard/no-op** agar notebook lama tetap mudah diikuti,
tetapi tidak lagi mengubah bobot model.


In [ ]:

print("=== [DINONAKTIFKAN] Fine-tune V2 target tetap ===")
print(
    "Cell V2 sengaja TIDAK menjalankan training. Training dengan target 'sabila' "
    "yang sama di setiap batch berisiko menyebabkan shortcut/collapse decoder."
)
print(
    "Gunakan model general-payload dari Bagian 5, jalankan [DIAGNOSTIK], lalu "
    "gunakan [FIX V3] yang melatih payload acak tanpa mengoptimasi khusus 'sabila'."
)

LEGACY_FT_V2_CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, 'checkpoint_finetune_v2.pth'
)
if os.path.exists(LEGACY_FT_V2_CHECKPOINT_PATH):
    print(
        "[INFO] Checkpoint V2 lama ditemukan tetapi DIABAIKAN:",
        LEGACY_FT_V2_CHECKPOINT_PATH
    )


## [WAJIB] Sanity Test Anti-Hafalan / Anti-Decoder-Collapse

Tes ini mengecek empat hal pada **video evaluation hold-out**:

1. frame tanpa watermark tidak boleh menyerupai target `"sabila"`;
2. target produksi harus terbaca jika benar-benar ditanam;
3. payload acak 144-bit harus ikut terbaca;
4. payload acak berstruktur ECC harus pulih setelah soft/hard vote.

Known-target yang tinggi tanpa random-payload generalization dianggap **gagal**, karena
itu adalah gejala decoder menghafal jawaban tetap.


In [ ]:
def run_sanity_checks(video_names, label='SANITY', max_videos=3, max_frames=16, trials=3):
    encoder.eval()
    decoder.eval()

    names = list(video_names)[:max_videos]
    if not names:
        raise RuntimeError("Tidak ada video evaluation untuk sanity test.")

    def aggregate_decode(frames_t):
        frame_logits = decoder(frames_t)
        # Rata-rata logit menggabungkan bukti lintas-frame sebelum sigmoid.
        aggregate_logits = frame_logits.mean(dim=0)
        aggregate_probs = torch.sigmoid(aggregate_logits)
        bits = (aggregate_probs >= 0.5).float()
        confidence = ((aggregate_probs - 0.5).abs() * 2.0).mean().item()
        return bits, aggregate_probs, frame_logits, confidence

    def raw_bit_acc(pred_bits, target_bits):
        pred = pred_bits.detach().reshape(-1)
        target = target_bits.detach().reshape(-1).to(pred.device)
        return (pred == target).float().mean().item()

    clean_target_accs = []
    clean_target_hits = 0
    clean_confidences = []
    target_accs = []
    random_general_accs = []
    random_ecc_raw_accs = []
    random_ecc_payload_accs = []

    with torch.no_grad():
        for video_name in names:
            frames = dataset_frames[video_name][0][:max_frames]
            frames_t = torch.tensor(
                np.stack(frames), dtype=torch.float32, device=device
            ).permute(0, 3, 1, 2)

            clean_bits, clean_probs, _, clean_conf = aggregate_decode(frames_t)
            clean_text = probabilities_to_text_ecc(clean_probs.cpu().numpy())
            clean_acc = raw_bit_acc(clean_bits, GROUND_TRUTH_WM[0])
            clean_target_accs.append(clean_acc)
            clean_target_hits += int(clean_text == WATERMARK_TEXT)
            clean_confidences.append(clean_conf)
            print(
                f"[{label} NEGATIVE] {video_name} | target_similarity={clean_acc:.4f} | "
                f"decoded='{clean_text}' | confidence={clean_conf:.3f}"
            )

            target_batch = GROUND_TRUTH_WM.repeat(frames_t.size(0), 1)
            target_frames, _ = encoder(frames_t, target_batch)
            target_bits, target_probs, _, _ = aggregate_decode(target_frames)
            target_accs.append(raw_bit_acc(target_bits, GROUND_TRUTH_WM[0]))

            for _ in range(trials):
                random_single = torch.randint(
                    0, 2, (1, WATERMARK_LENGTH), device=device
                ).float()
                random_frames, _ = encoder(
                    frames_t, random_single.repeat(frames_t.size(0), 1)
                )
                pred_random, _, _, _ = aggregate_decode(random_frames)
                random_general_accs.append(raw_bit_acc(pred_random, random_single[0]))

                ecc_single, raw_payload = make_random_ecc_watermark(1)
                ecc_frames, _ = encoder(
                    frames_t, ecc_single.repeat(frames_t.size(0), 1)
                )
                pred_ecc, prob_ecc, _, _ = aggregate_decode(ecc_frames)
                random_ecc_raw_accs.append(raw_bit_acc(pred_ecc, ecc_single[0]))

                voted_payload, _ = ecc_soft_vote(prob_ecc.cpu().numpy())
                random_ecc_payload_accs.append(float(
                    (voted_payload == raw_payload[0].cpu().numpy()).mean()
                ))

    summary = {
        'label': label,
        'clean_target_similarity': float(np.mean(clean_target_accs)),
        'clean_target_hits': int(clean_target_hits),
        'clean_confidence': float(np.mean(clean_confidences)),
        'target_raw_acc': float(np.mean(target_accs)),
        'random144_raw_acc': float(np.mean(random_general_accs)),
        'random_ecc_raw_acc': float(np.mean(random_ecc_raw_accs)),
        'random_ecc_payload_acc': float(np.mean(random_ecc_payload_accs)),
    }
    summary['passed'] = bool(
        summary['clean_target_hits'] == 0
        and summary['clean_target_similarity'] < 0.70
        and summary['target_raw_acc'] >= 0.90
        and summary['random144_raw_acc'] >= 0.80
        and summary['random_ecc_payload_acc'] >= 0.90
    )

    print(f"\n=== {label} SUMMARY ===")
    for key, value in summary.items():
        print(f"{key:28}: {value}")
    print("PASS" if summary['passed'] else "FAIL — model belum boleh dipakai untuk evaluasi codec")
    return summary


PRE_TRAIN_SANITY = run_sanity_checks(eval_video_names, label='PRE-V3')



## [PHASE-1 FIX V3] General-Payload Fine-tune Anti-Collapse

Fine-tune V3 sekarang **tidak mengoptimasi `"sabila"` sama sekali**.

Setiap clip memperoleh satu dari dua jenis target:

- 144 bit acak bebas; atau
- 48 bit payload acak yang diulang 3× mengikuti struktur ECC produksi.

`"sabila"` hanya dipakai sebagai **validation payload**, tanpa gradient. Dengan demikian
akurasi `"sabila"` tidak bisa diperoleh hanya karena target itu terus-menerus muncul saat
training.


In [ ]:

print("=== [PHASE-1 FIX V3] General-payload fine-tune anti-collapse ===")
print(
    "Training TIDAK memakai 'sabila' sebagai target gradient. "
    "'sabila' hanya digunakan untuk validasi."
)

FT3_EPOCHS = 60
FT3_CLEAN_WARMUP_EPOCHS = 5
FT3_LR = 3e-4
FT3_GENERAL_RANDOM_PROB = 0.50   # sisanya random ECC
FT3_CHECKPOINT_EVERY = 10
FT3_CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, 'checkpoint_finetune_v5_group_split.pth'
)

MAIN_ENCODER_PATH = os.path.join(OUTPUT_DIR, 'encoder.pth')
MAIN_DECODER_PATH = os.path.join(OUTPUT_DIR, 'decoder.pth')

ft3_history = {
    'loss': [],
    'bit_acc_random144': [],
    'bit_acc_random_ecc': [],
    'bit_acc_sabila_eval': [],
}
ft3_start_epoch = 1

# Optimizer baru: jangan mewarisi momentum dari legacy fine-tune target tetap.
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=FT3_LR
)

if os.path.exists(FT3_CHECKPOINT_PATH):
    ft3_ckpt = torch.load(FT3_CHECKPOINT_PATH, map_location=device)
    encoder.load_state_dict(ft3_ckpt['encoder'])
    decoder.load_state_dict(ft3_ckpt['decoder'])
    optimizer.load_state_dict(ft3_ckpt['optimizer'])
    ft3_history = ft3_ckpt['history']
    ft3_start_epoch = ft3_ckpt['epoch'] + 1
    print(f"[RESUME V3] melanjutkan dari epoch {ft3_start_epoch}.")
else:
    # Selalu mulai V3 dari model general-payload Bagian 5 bila file tersedia.
    # Jangan mulai dari checkpoint Fine-tune V2 yang mungkin sudah collapse.
    if os.path.exists(MAIN_ENCODER_PATH) and os.path.exists(MAIN_DECODER_PATH):
        encoder.load_state_dict(torch.load(MAIN_ENCODER_PATH, map_location=device))
        decoder.load_state_dict(torch.load(MAIN_DECODER_PATH, map_location=device))
        print("[INIT V3] Memuat encoder.pth + decoder.pth dari training general-payload Bagian 5.")
    else:
        print(
            "[INIT V3] File model Bagian 5 belum ditemukan; memakai bobot encoder/decoder "
            "yang saat ini ada di memori. Pastikan Cell V2 legacy TIDAK dijalankan."
        )

# Pilih validation video yang tidak masuk train jika tersedia.
_holdout_names = [v for v in dataset_frames.keys() if v not in set(train_video_names)]
if _holdout_names:
    FT3_VAL_VIDEO = _holdout_names[0]
    print(f"[VALIDATION] Hold-out video: {FT3_VAL_VIDEO}")
else:
    FT3_VAL_VIDEO = list(dataset_frames.keys())[-1]
    print(
        "[VALIDATION WARNING] Tidak ada video hold-out di dataset_frames; "
        f"menggunakan '{FT3_VAL_VIDEO}' hanya sebagai sanity validation."
    )

def ft3_save_checkpoint(epoch):
    torch.save({
        'encoder': encoder.state_dict(),
        'decoder': decoder.state_dict(),
        'optimizer': optimizer.state_dict(),
        'history': ft3_history,
        'epoch': epoch,
    }, FT3_CHECKPOINT_PATH)


def _ft3_eval_sabila_lossless():
    """Validation only — tidak ada backward/optimizer terhadap 'sabila'."""
    encoder.eval()
    decoder.eval()
    frames, _, _ = dataset_frames[FT3_VAL_VIDEO]
    frames = frames[:min(16, len(frames))]
    frames_t = torch.tensor(
        np.stack(frames), dtype=torch.float32, device=device
    ).permute(0, 3, 1, 2)

    with torch.no_grad():
        wm_batch = GROUND_TRUTH_WM.repeat(frames_t.size(0), 1)
        watermarked, _ = encoder(frames_t, wm_batch)
        probs = torch.sigmoid(decoder(watermarked)).mean(dim=0)
        pred = (probs > 0.5).float()
        return (pred == GROUND_TRUTH_WM[0]).float().mean().item()


stable_pass_epochs = 0

for epoch in range(ft3_start_epoch, FT3_EPOCHS + 1):
    encoder.train()
    decoder.train()

    epoch_loss = 0.0
    total_samples = 0
    correct_random144 = total_random144 = 0
    correct_random_ecc = total_random_ecc = 0

    for vname in train_video_names:
        frames, fps, size = dataset_frames[vname]
        frames_np = np.stack(frames)
        n_local = frames_np.shape[0]

        starts = list(range(0, max(n_local - BATCH_SIZE, 0) + 1, BATCH_SIZE))
        if not starts:
            starts = [0]
        random.shuffle(starts)

        for start in starts:
            idx_local = np.arange(start, min(start + BATCH_SIZE, n_local))
            batch = torch.tensor(
                frames_np[idx_local], dtype=torch.float32, device=device
            ).permute(0, 3, 1, 2)
            batch = augment_batch(batch)
            B = batch.size(0)

            # SATU payload konsisten untuk seluruh frame dalam clip.
            use_general_random = random.random() < FT3_GENERAL_RANDOM_PROB
            if use_general_random:
                wm_single = torch.randint(
                    0, 2, (1, WATERMARK_LENGTH), device=device
                ).float()
                target_kind = 'random144'
            else:
                wm_single, _ = make_random_ecc_watermark(1)
                target_kind = 'random_ecc'

            wm_bits = wm_single.repeat(B, 1)
            watermarked, residual = encoder(batch, wm_bits)

            # Lima epoch awal menguatkan kanal watermark lossless lebih dulu.
            if epoch <= FT3_CLEAN_WARMUP_EPOCHS:
                noised = watermarked
                noise_mode = 'none'
            else:
                noised, noise_mode = apply_curriculum_noise(
                    watermarked,
                    epoch=WARMUP_EPOCHS + CRF_CURRICULUM_EPOCHS,
                    total_epochs=EPOCHS,
                    prev_bit_acc=1.0,
                )

            pred_logits = decoder(noised)

            loss_img = mse_loss(watermarked, batch)
            loss_wm = bce_loss(pred_logits, wm_bits)
            loss_bias = residual.mean(dim=(2, 3)).pow(2).mean()

            # Watermark tetap prioritas; image/bias menjaga invisibility.
            loss = 0.5 * loss_img + 1.0 * loss_wm + 2.0 * loss_bias

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()),
                max_norm=5.0
            )
            optimizer.step()

            with torch.no_grad():
                pred_bits = (torch.sigmoid(pred_logits) > 0.5).float()
                n_correct = (pred_bits == wm_bits).sum().item()

                if target_kind == 'random144':
                    correct_random144 += n_correct
                    total_random144 += wm_bits.numel()
                else:
                    correct_random_ecc += n_correct
                    total_random_ecc += wm_bits.numel()

            epoch_loss += loss.item() * B
            total_samples += B

            del batch, watermarked, noised, pred_logits

        del frames, frames_np

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    avg_loss = epoch_loss / max(total_samples, 1)
    acc_random144 = correct_random144 / max(total_random144, 1)
    acc_random_ecc = correct_random_ecc / max(total_random_ecc, 1)
    acc_sabila_eval = _ft3_eval_sabila_lossless()

    ft3_history['loss'].append(avg_loss)
    ft3_history['bit_acc_random144'].append(acc_random144)
    ft3_history['bit_acc_random_ecc'].append(acc_random_ecc)
    ft3_history['bit_acc_sabila_eval'].append(acc_sabila_eval)

    print(
        f"[V3] Epoch {epoch:02d}/{FT3_EPOCHS} | loss={avg_loss:.5f} | "
        f"random144={acc_random144:.4f} | randomECC={acc_random_ecc:.4f} | "
        f"sabila(eval-only)={acc_sabila_eval:.4f}"
    )

    if (
        acc_random144 >= 0.90
        and acc_random_ecc >= 0.90
        and acc_sabila_eval >= 0.95
    ):
        stable_pass_epochs += 1
    else:
        stable_pass_epochs = 0

    if epoch % FT3_CHECKPOINT_EVERY == 0:
        ft3_save_checkpoint(epoch)
        print(f"   [checkpoint V3 tersimpan @ epoch {epoch}]")

    # Hindari berhenti karena satu epoch kebetulan bagus.
    if stable_pass_epochs >= 3 and epoch > FT3_CLEAN_WARMUP_EPOCHS:
        print(
            "[EARLY STOP] Tiga epoch berturut-turut lolos target generalisasi. "
            "Lanjutkan dengan menjalankan ulang Cell [DIAGNOSTIK]."
        )
        ft3_save_checkpoint(epoch)
        break

torch.save(
    encoder.state_dict(),
    os.path.join(OUTPUT_DIR, 'encoder_finetune_v3.pth')
)
torch.save(
    decoder.state_dict(),
    os.path.join(OUTPUT_DIR, 'decoder_finetune_v3.pth')
)

print(
    "\n[SELESAI V3] Jalankan ulang Cell [PHASE-1 DIAGNOSTIK]. "
    "PASS hanya valid jika random payload ikut tinggi dan negative control bersih."
)

plt.figure(figsize=(8, 4))
plt.plot(ft3_history['bit_acc_random144'], label='random 144-bit')
plt.plot(ft3_history['bit_acc_random_ecc'], label='random ECC')
plt.plot(ft3_history['bit_acc_sabila_eval'], label="'sabila' eval-only")
plt.axhline(0.90, linestyle='--', label='generalization target 0.90')
plt.xlabel('Epoch')
plt.ylabel('Bit accuracy')
plt.title('Fine-tune V3 — Generalization, bukan hafalan target')
plt.legend()
plt.tight_layout()
plt.show()

# Wajib dijalankan otomatis setelah V3. Hasil PRE-V3 tidak dipakai sebagai gate final.
POST_TRAIN_SANITY = run_sanity_checks(eval_video_names, label='POST-V3')
SANITY_PASS = bool(POST_TRAIN_SANITY['passed'])
if not SANITY_PASS:
    print(
        "\n[STOP] Model belum lolos generalization gate. Naikkan FT3_EPOCHS, "
        "jalankan ulang V3, lalu ulangi gate sebelum evaluasi codec."
    )



## 6. Penyisipan Watermark ke Video

Bagian ini sekarang menggunakan `GROUND_TRUTH_WM` ECC yang sudah dibuat sejak awal.
Tidak ada lagi definisi ulang payload 64/144 bit dan tidak ada lagi penyisipan kedua kali
di cell `[FIX]` berikutnya.


In [ ]:

if not globals().get('SANITY_PASS', False):
    raise RuntimeError(
        'Evaluasi dihentikan: model belum lolos POST-V3 sanity/generalization gate.'
    )

encoder.eval()

# Alias kompatibilitas untuk cell evaluasi lama.
text_to_bits = lambda text, bit_length=WATERMARK_LENGTH: (
    text_to_bits_ecc(text, ECC_REPEAT, bit_length)[0]
)
bits_to_text = lambda bits: bits_to_text_ecc(
    bits, PAYLOAD_BITS, ECC_REPEAT
)

assert GROUND_TRUTH_WM.shape == (1, WATERMARK_LENGTH)
assert bits_to_text(
    GROUND_TRUTH_WM[0].detach().cpu().numpy()
) == WATERMARK_TEXT

print(f"Watermark produksi : '{WATERMARK_TEXT}'")
print(
    f"Payload/ECC        : {PAYLOAD_BITS} bit x {ECC_REPEAT} "
    f"= {WATERMARK_LENGTH} bit"
)

watermarked_videos = {}

with torch.no_grad():
    for vname in eval_video_names:
        frames, fps, orig_size = dataset_frames[vname]
        frames_t = torch.tensor(
            np.stack(frames), dtype=torch.float32, device=device
        ).permute(0, 3, 1, 2)

        # SATU payload yang sama untuk semua frame video.
        wm_batch = GROUND_TRUTH_WM.repeat(frames_t.size(0), 1)
        watermarked_t, residual = encoder(frames_t, wm_batch)

        watermarked_np = (
            watermarked_t.permute(0, 2, 3, 1).cpu().numpy()
        )

        out_name = os.path.splitext(vname)[0] + '_watermarked.mkv'
        out_path = str(WATERMARKED_DIR / out_name)

        # Simpan lossless supaya tahap kompresi hanya mendapat SATU sumber
        # distorsi terukur: codec yang memang sedang diuji.
        frames_to_video_lossless(
            list(watermarked_np),
            out_path,
            fps,
            size=FRAME_SIZE[::-1]
        )
        watermarked_videos[vname] = out_path

        print(
            f"[WATERMARK ECC] {vname} -> {out_path} | "
            f"identity='{WATERMARK_TEXT}'"
        )



## [PHASE-1 FIX] Verifikasi Ground Truth ECC

Versi lama cell ini menyisipkan ulang seluruh dataset karena Bagian 6 sebelumnya sempat
memakai payload yang salah. Setelah Bagian 6 diperbaiki, re-insertion tidak lagi diperlukan.

Cell berikut hanya melakukan assertion/verifikasi state agar tidak ada kompresi ganda atau
hasil yang bergantung pada urutan eksekusi notebook.


In [ ]:

print("=== [PHASE-1 FIX] Verifikasi state watermark ECC ===")

expected_bits, expected_payload_bits = text_to_bits_ecc(
    WATERMARK_TEXT, ECC_REPEAT, WATERMARK_LENGTH
)
expected_tensor = torch.tensor(
    [expected_bits], dtype=torch.float32, device=device
)

assert expected_payload_bits == PAYLOAD_BITS
assert expected_tensor.shape == GROUND_TRUTH_WM.shape
assert torch.equal(expected_tensor, GROUND_TRUTH_WM)
assert bits_to_text_ecc(
    GROUND_TRUTH_WM[0].detach().cpu().numpy(),
    PAYLOAD_BITS,
    ECC_REPEAT
) == WATERMARK_TEXT

if 'watermarked_videos' not in globals() or not watermarked_videos:
    raise RuntimeError(
        "watermarked_videos belum tersedia. Jalankan Bagian 6 terlebih dahulu."
    )

print(f"Ground truth ECC valid : '{WATERMARK_TEXT}'")
print(f"Jumlah video tersisip  : {len(watermarked_videos)}")
print("Tidak ada penyisipan ulang — output Bagian 6 dipakai langsung.")


## 7. Pengujian Kompresi Video (H.264, H.265, Neural Codec)

Setiap video ber-watermark dikompresi menggunakan **tiga metode**:

- **H.264** dan **H.265** — menggunakan `ffmpeg` (codec konvensional standar industri).
- **Neural Codec** — menggunakan *compressive autoencoder* sederhana (CNN dengan bottleneck + kuantisasi), sebagai representasi codec berbasis neural network.


**Catatan:** model Neural Codec sudah dilatih di Bagian 4B (dan bahkan sudah ikut serta saat training watermark). Sel di bawah ini hanya memakainya untuk menghasilkan video uji terkompresi Neural Codec.

In [ ]:
import subprocess

def compress_ffmpeg(input_path, output_path, codec='libx264', crf=35):
    """Kompresi H.264/H.265 nyata dengan konfigurasi evaluasi konsisten."""
    command = [
        'ffmpeg', '-y', '-loglevel', 'error', '-i', str(input_path),
        '-an', '-c:v', codec, '-crf', str(crf), '-preset', 'fast',
        '-pix_fmt', 'yuv420p', str(output_path),
    ]
    subprocess.run(command, check=True)
    if not Path(output_path).is_file() or Path(output_path).stat().st_size == 0:
        raise RuntimeError(f"Output ffmpeg tidak valid: {output_path}")
    return str(output_path)


CRF_HIGH_COMPRESSION = 35
compressed_paths = {}
compression_artifact_paths = {}

for video_name, watermarked_path in watermarked_videos.items():
    stem = Path(video_name).stem
    h264_path = COMPRESSED_DIR / f'{stem}_h264.mp4'
    h265_path = COMPRESSED_DIR / f'{stem}_h265.mp4'

    compress_ffmpeg(watermarked_path, h264_path, codec='libx264', crf=CRF_HIGH_COMPRESSION)
    compress_ffmpeg(watermarked_path, h265_path, codec='libx265', crf=CRF_HIGH_COMPRESSION)

    compressed_paths[video_name] = {
        'h264': str(h264_path),
        'h265': str(h265_path),
    }
    compression_artifact_paths[video_name] = {
        'h264': str(h264_path),
        'h265': str(h265_path),
    }
    print(f"{video_name}: H.264/H.265 CRF {CRF_HIGH_COMPRESSION} selesai.")


def compress_neural_codec(frames, fps, reconstructed_path, bitstream_path, quant_levels=16):
    """Encode latent terkuantisasi lalu rekonstruksi frame.

    `bitstream_path` adalah artefak yang dipakai untuk menghitung ukuran kompresi.
    `reconstructed_path` hanya dipakai decoder watermark serta PSNR/SSIM.
    """
    frames_t = torch.tensor(
        np.stack(frames), dtype=torch.float32, device=device
    ).permute(0, 3, 1, 2)

    with torch.no_grad():
        latent = torch.tanh(neural_codec.enc(frames_t))
        latent_q = torch.round(latent * quant_levels).clamp(
            -quant_levels, quant_levels
        ).to(torch.int8)
        latent_hat = latent_q.float() / quant_levels
        reconstruction = torch.sigmoid(neural_codec.dec(latent_hat))

    latent_np = latent_q.cpu().numpy()
    np.savez_compressed(
        bitstream_path,
        latent=latent_np,
        fps=np.float32(fps),
        quant_levels=np.int16(quant_levels),
        frame_height=np.int16(FRAME_SIZE[0]),
        frame_width=np.int16(FRAME_SIZE[1]),
    )

    reconstruction_np = reconstruction.permute(0, 2, 3, 1).cpu().numpy()
    frames_to_video_lossless(
        list(reconstruction_np),
        reconstructed_path,
        fps,
        size=FRAME_SIZE[::-1],
    )
    return str(reconstructed_path), str(bitstream_path)


for video_name, watermarked_path in watermarked_videos.items():
    frames_wm, fps_wm, _ = extract_frames(watermarked_path)
    stem = Path(video_name).stem
    reconstructed_path = COMPRESSED_DIR / f'{stem}_neuralcodec_reconstructed.mkv'
    bitstream_path = COMPRESSED_DIR / f'{stem}_neuralcodec_latent.npz'

    reconstructed_path, bitstream_path = compress_neural_codec(
        frames_wm,
        fps_wm,
        reconstructed_path,
        bitstream_path,
    )
    compressed_paths[video_name]['neural'] = reconstructed_path
    compression_artifact_paths[video_name]['neural'] = bitstream_path
    print(
        f"{video_name}: neural codec selesai | reconstruction={reconstructed_path} | "
        f"latent={bitstream_path}"
    )



## 8. Deteksi Watermark — Raw Bit + Diagnostik Per-Frame

Hasil video tetap diagregasi dengan rata-rata probabilitas antar frame, tetapi sekarang
disimpan juga diagnostik **akurasi per-frame**, **akurasi agregat**, dan **confidence**.

Hal ini mencegah hasil agregat 100% menutupi kenyataan bahwa mayoritas frame individual
sebenarnya lemah.


In [ ]:

decoder.eval()

def extract_watermark_with_diagnostics(video_path, ground_truth_bits=None):
    frames, fps, size = extract_frames(video_path)
    if not frames:
        raise RuntimeError(f"Tidak ada frame yang bisa dibaca: {video_path}")

    frames_t = torch.tensor(
        np.stack(frames), dtype=torch.float32, device=device
    ).permute(0, 3, 1, 2)

    with torch.no_grad():
        frame_logits = decoder(frames_t)
        frame_probs = torch.sigmoid(frame_logits)

    aggregate_logits = frame_logits.mean(dim=0)
    avg_probs = torch.sigmoid(aggregate_logits)
    extracted_bits = (avg_probs >= 0.5).float()

    details = {
        'n_frames': int(frames_t.size(0)),
        'aggregate_confidence': float(
            ((avg_probs - 0.5).abs() * 2.0).mean().item()
        ),
    }

    if ground_truth_bits is not None:
        gt = torch.as_tensor(
            ground_truth_bits,
            dtype=torch.float32,
            device=device
        ).reshape(1, -1)

        frame_bits = (frame_probs > 0.5).float()
        frame_acc = (frame_bits == gt).float().mean(dim=1)

        details.update({
            'frame_acc_mean': float(frame_acc.mean().item()),
            'frame_acc_min': float(frame_acc.min().item()),
            'frame_acc_max': float(frame_acc.max().item()),
            'aggregate_raw_acc': float(
                (extracted_bits.reshape(1, -1) == gt).float().mean().item()
            ),
        })

    return (
        extracted_bits.detach().cpu().numpy(),
        avg_probs.detach().cpu().numpy(),
        details,
    )


# Wrapper kompatibel dengan cell lama.
def extract_watermark_from_video(video_path):
    bits, probs, _ = extract_watermark_with_diagnostics(video_path)
    return bits, probs


extraction_results = {}
extraction_diagnostics = {}
gt_for_eval = GROUND_TRUTH_WM[0].detach().cpu().numpy()

for vname in watermarked_videos.keys():
    extraction_results[vname] = {}
    extraction_diagnostics[vname] = {}

    for codec_name, path in compressed_paths[vname].items():
        bits, probs, details = extract_watermark_with_diagnostics(
            path,
            ground_truth_bits=gt_for_eval
        )

        extraction_results[vname][codec_name] = (bits, probs)
        extraction_diagnostics[vname][codec_name] = details

        print(
            f"{vname} [{codec_name}] | "
            f"agg_acc={details['aggregate_raw_acc']:.4f} | "
            f"frame_acc(mean/min/max)="
            f"{details['frame_acc_mean']:.4f}/"
            f"{details['frame_acc_min']:.4f}/"
            f"{details['frame_acc_max']:.4f} | "
            f"confidence={details['aggregate_confidence']:.4f}"
        )


## 8A. Ekstraksi Teks: Raw BER dan Soft ECC Recovery

Decoder menghasilkan 144 probabilitas (tiga salinan dari payload 48-bit). Laporan
memisahkan:

- **raw BER** sebelum ECC;
- teks setelah probabilitas tiga salinan digabung dengan **soft vote**;
- kecocokan teks persis dengan `"sabila"`.

Pemisahan ini penting agar keberhasilan ECC tidak menyembunyikan error mentah model.


In [ ]:
import pandas as pd

print("=== Decode hasil ekstraksi: raw BER -> soft ECC -> teks ===")

gt_bits_eval = GROUND_TRUTH_WM[0].detach().cpu().numpy().reshape(-1)
text_recovery_records = []

for video_name, per_codec in extraction_results.items():
    for codec_name, (bits, probabilities) in per_codec.items():
        predicted_bits = np.asarray(bits).reshape(-1)
        raw_ber = float(np.mean(predicted_bits != gt_bits_eval))
        decoded_text = probabilities_to_text_ecc(probabilities)
        exact_match = decoded_text == WATERMARK_TEXT

        text_recovery_records.append({
            'video': video_name,
            'codec': codec_name,
            'raw_ber': raw_ber,
            'raw_bit_accuracy': 1.0 - raw_ber,
            'ecc_text': decoded_text,
            'exact_text_match': exact_match,
        })
        print(
            f"{video_name} [{codec_name}] | raw_BER={raw_ber:.4f} | "
            f"ECC_text='{decoded_text}' | exact_match={exact_match}"
        )

df_text_recovery = pd.DataFrame(text_recovery_records)



## 8B. Validasi Lossless — Target Tetap **dan** Payload Acak

Baseline lossless sekarang mempunyai control yang tidak dapat dilewati dengan menghafal
`"sabila"`:

- seluruh video diuji dengan target produksi `"sabila"`;
- subset video diuji dengan beberapa **random 144-bit payload**;
- subset yang sama diuji dengan **random ECC payload**.

Jika `"sabila"` tinggi tetapi payload acak rendah, encoder-decoder belum dianggap sehat.


In [ ]:

import pandas as pd

encoder.eval()
decoder.eval()

LOSSLESS_RANDOM_VIDEO_LIMIT = 5
LOSSLESS_RANDOM_TRIALS = 3
LOSSLESS_MAX_FRAMES_PER_VIDEO = 32

gt_bits_eval = GROUND_TRUTH_WM[0].detach().cpu().numpy().reshape(-1)

lossless_records = []
random_control_records = []

with torch.no_grad():
    for video_index, vname in enumerate(eval_video_names):
        frames, fps, orig_size = dataset_frames[vname]
        frames_subset = frames[:LOSSLESS_MAX_FRAMES_PER_VIDEO]
        frames_t = torch.tensor(
            np.stack(frames_subset),
            dtype=torch.float32,
            device=device
        ).permute(0, 3, 1, 2)

        # ----------------------------------------------------------
        # A. Target produksi 'sabila'
        # ----------------------------------------------------------
        wm_batch = GROUND_TRUTH_WM.repeat(frames_t.size(0), 1)
        watermarked_t, _ = encoder(frames_t, wm_batch)

        frame_probs = torch.sigmoid(decoder(watermarked_t))
        avg_probs = frame_probs.mean(dim=0)
        bits_pred = (avg_probs > 0.5).float().cpu().numpy()

        raw_ber = float(np.mean(gt_bits_eval != bits_pred))
        decoded_text = bits_to_text_ecc(
            bits_pred, PAYLOAD_BITS, ECC_REPEAT
        )

        frame_bits = (frame_probs > 0.5).float()
        frame_acc = (
            frame_bits == GROUND_TRUTH_WM
        ).float().mean(dim=1)

        lossless_records.append({
            'video': vname,
            'raw_ber_lossless': raw_ber,
            'raw_bit_acc_lossless': 1.0 - raw_ber,
            'frame_acc_mean': float(frame_acc.mean().item()),
            'frame_acc_min': float(frame_acc.min().item()),
            'teks_terekstrak': decoded_text,
            'exact_text_match': decoded_text == WATERMARK_TEXT,
        })

        # ----------------------------------------------------------
        # B. Random controls — cukup subset video agar cepat.
        # ----------------------------------------------------------
        if video_index < LOSSLESS_RANDOM_VIDEO_LIMIT:
            for trial in range(LOSSLESS_RANDOM_TRIALS):
                # 144-bit random bebas
                random_single = torch.randint(
                    0, 2, (1, WATERMARK_LENGTH), device=device
                ).float()
                random_batch = random_single.repeat(frames_t.size(0), 1)
                wm_random, _ = encoder(frames_t, random_batch)

                p_random = torch.sigmoid(decoder(wm_random)).mean(dim=0)
                b_random = (p_random > 0.5).float()
                random_acc = (
                    b_random == random_single[0]
                ).float().mean().item()

                random_control_records.append({
                    'video': vname,
                    'trial': trial + 1,
                    'jenis': 'random144',
                    'raw_bit_acc': random_acc,
                    'payload_vote_acc': np.nan,
                })

                # Random 48-bit payload dengan struktur ECC produksi
                ecc_single, raw_payload = make_random_ecc_watermark(1)
                ecc_batch = ecc_single.repeat(frames_t.size(0), 1)
                wm_ecc, _ = encoder(frames_t, ecc_batch)

                p_ecc = torch.sigmoid(decoder(wm_ecc)).mean(dim=0)
                b_ecc = (p_ecc > 0.5).float()

                ecc_raw_acc = (
                    b_ecc == ecc_single[0]
                ).float().mean().item()

                voted = ecc_majority_vote(
                    b_ecc.cpu().numpy(),
                    PAYLOAD_BITS,
                    ECC_REPEAT
                )
                payload_vote_acc = float(
                    (voted == raw_payload[0].cpu().numpy()).mean()
                )

                random_control_records.append({
                    'video': vname,
                    'trial': trial + 1,
                    'jenis': 'randomECC',
                    'raw_bit_acc': ecc_raw_acc,
                    'payload_vote_acc': payload_vote_acc,
                })


df_lossless = pd.DataFrame(lossless_records)
df_random_lossless = pd.DataFrame(random_control_records)

print(f"Ground truth teks : '{WATERMARK_TEXT}'")
print(
    "Target mean raw bit_acc (lossless) : "
    f"{df_lossless['raw_bit_acc_lossless'].mean():.4f}"
)
print(
    "Target exact text recovery          : "
    f"{df_lossless['exact_text_match'].sum()}/{len(df_lossless)}"
)

if not df_random_lossless.empty:
    print("\nRandom-control mean accuracy:")
    display(
        df_random_lossless.groupby('jenis', dropna=False)[
            ['raw_bit_acc', 'payload_vote_acc']
        ].mean().reset_index()
    )

display(df_lossless)

target_mean = float(df_lossless['raw_bit_acc_lossless'].mean())
random144_mean = float(
    df_random_lossless.loc[
        df_random_lossless['jenis'] == 'random144',
        'raw_bit_acc'
    ].mean()
)
random_ecc_payload_mean = float(
    df_random_lossless.loc[
        df_random_lossless['jenis'] == 'randomECC',
        'payload_vote_acc'
    ].mean()
)

print("\n=== LOSSLESS GATE ===")
if target_mean >= 0.95 and random144_mean >= 0.90 and random_ecc_payload_mean >= 0.90:
    print(
        "[PASS] Encoder-decoder lossless menunjukkan generalisasi payload. "
        "Baru layak melanjutkan analisis robustness codec."
    )
elif target_mean >= 0.95 and random144_mean < 0.75:
    print(
        "[FAIL - SHORTCUT SUSPECTED] Target 'sabila' tinggi tetapi payload acak rendah."
    )
else:
    print(
        "[FAIL] Kanal watermark lossless belum cukup stabil. "
        "Perbaiki training sebelum menyimpulkan masalah berasal dari codec."
    )



## 8C. Analisis Akurasi per Posisi Bit ECC

Tidak ada lagi konsep `padding 16 bit`. Semua 144 posisi adalah bagian dari tiga salinan
payload ECC: copy-1, copy-2, dan copy-3. Analisis di bawah menandai **nomor salinan ECC**
dan **posisi payload di dalam salinan**.


In [ ]:

gt_arr = GROUND_TRUTH_WM[0].detach().cpu().numpy().reshape(-1)

all_pred_bits = []
all_pred_meta = []

for vname, per_codec in extraction_results.items():
    for codec_name, (bits_pred, probs) in per_codec.items():
        all_pred_bits.append(np.asarray(bits_pred).reshape(-1))
        all_pred_meta.append((vname, codec_name))

all_pred_bits = np.stack(all_pred_bits)
per_bit_acc = (all_pred_bits == gt_arr).mean(axis=0)

df_per_bit = pd.DataFrame({
    'bit_ke_global': np.arange(1, WATERMARK_LENGTH + 1),
    'ecc_copy': (np.arange(WATERMARK_LENGTH) // PAYLOAD_BITS) + 1,
    'payload_bit_ke': (np.arange(WATERMARK_LENGTH) % PAYLOAD_BITS) + 1,
    'ground_truth': gt_arr.astype(int),
    'akurasi': per_bit_acc,
})

print(f"Jumlah sampel video x codec : {len(all_pred_bits)}")
print(f"Raw bit accuracy keseluruhan: {per_bit_acc.mean():.4f}\n")

print("10 posisi bit PALING SERING SALAH:")
display(
    df_per_bit.sort_values('akurasi').head(10)
)

print("\nAkurasi rata-rata per salinan ECC:")
display(
    df_per_bit.groupby('ecc_copy', as_index=False)['akurasi'].mean()
)

plt.figure(figsize=(14, 4))
plt.bar(df_per_bit['bit_ke_global'], df_per_bit['akurasi'])
for boundary in range(1, ECC_REPEAT):
    plt.axvline(boundary * PAYLOAD_BITS + 0.5, linestyle='--')
plt.axhline(0.95, linestyle='--', label='0.95')
plt.xlabel('Posisi bit global')
plt.ylabel('Akurasi')
plt.title('Akurasi Ekstraksi per Posisi Bit — tiga salinan ECC')
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'akurasi_per_posisi_bit_ecc.png'),
    dpi=150
)
plt.show()



## Analisis ECC: Sebelum dan Sesudah Majority Vote

Cell berikut membandingkan akurasi setiap salinan ECC dengan keberhasilan payload setelah
majority vote. Ini adalah tempat yang benar untuk menunjukkan apakah ECC benar-benar
menolong ketika raw bit mengalami error.


In [ ]:

print("=== Analisis kontribusi ECC ===")

for k in range(ECC_REPEAT):
    lo = k * PAYLOAD_BITS
    hi = (k + 1) * PAYLOAD_BITS
    seg = per_bit_acc[lo:hi]
    print(
        f"Copy {k + 1}/{ECC_REPEAT} | bit {lo + 1}-{hi} | "
        f"mean raw accuracy={seg.mean():.4f}"
    )

gt_payload = ecc_majority_vote(
    gt_arr,
    PAYLOAD_BITS,
    ECC_REPEAT
)

raw_ber_values = []
post_vote_payload_acc = []
exact_text_recovery = 0

for bits_pred in all_pred_bits:
    raw_ber_values.append(float(np.mean(bits_pred != gt_arr)))

    voted_payload = ecc_majority_vote(
        bits_pred,
        PAYLOAD_BITS,
        ECC_REPEAT
    )
    payload_acc = float((voted_payload == gt_payload).mean())
    post_vote_payload_acc.append(payload_acc)

    decoded = bits_to_text_ecc(
        bits_pred,
        PAYLOAD_BITS,
        ECC_REPEAT
    )
    exact_text_recovery += int(decoded == WATERMARK_TEXT)

print(f"\nMean RAW BER sebelum ECC       : {np.mean(raw_ber_values):.4f}")
print(
    "Mean payload accuracy sesudah vote: "
    f"{np.mean(post_vote_payload_acc):.4f}"
)
print(
    f"Exact text '{WATERMARK_TEXT}' sesudah ECC : "
    f"{exact_text_recovery}/{len(all_pred_bits)} "
    f"({exact_text_recovery / len(all_pred_bits):.1%})"
)


## 9. Pengukuran Metrik Verifikasi Dasar (BER & Compression Ratio)

Sistem membandingkan watermark hasil ekstraksi dengan watermark asli (*ground truth*) untuk memperoleh **Bit Error Rate (BER)**, serta menghitung **Compression Ratio** dari rasio ukuran file terkompresi terhadap file ber-watermark.

Bagian ini **hanya mengukur** — tidak ada ambang dan tidak ada keputusan di sini. Seluruh penentuan status *Terdeteksi / Tidak Terdeteksi* dilakukan oleh metode usulan **CACS** pada Bagian 10B.

In [ ]:
def estimate_size_ratio(reference_path, compressed_artifact_path):
    """Ukuran artefak terkompresi / ukuran file video sumber; semakin kecil semakin padat.

    Untuk H.264/H.265, artefak adalah file MP4. Untuk neural codec, artefak adalah
    latent int8 terkuantisasi yang disimpan terkompresi dalam NPZ, bukan file FFV1
    hasil rekonstruksi.
    """
    source_bytes = Path(reference_path).stat().st_size
    compressed_bytes = Path(compressed_artifact_path).stat().st_size
    return compressed_bytes / max(source_bytes, 1)


def bit_error_rate(bits_true, bits_pred):
    bits_true = np.asarray(bits_true).reshape(-1)
    bits_pred = np.asarray(bits_pred).reshape(-1)
    return float(np.mean(bits_true != bits_pred))


gt_bits = GROUND_TRUTH_WM[0].detach().cpu().numpy().reshape(-1)
verification_records = []

for video_name in watermarked_videos:
    original_source_path = VIDEO_PATHS[video_name]
    for codec_name, reconstruction_path in compressed_paths[video_name].items():
        bits_pred, probabilities = extraction_results[video_name][codec_name]
        ber = bit_error_rate(gt_bits, bits_pred)
        artifact_path = compression_artifact_paths[video_name][codec_name]
        size_ratio = estimate_size_ratio(original_source_path, artifact_path)

        verification_records.append({
            'video': video_name,
            'codec': codec_name,
            'compression_ratio': size_ratio,
            'ber': ber,
            'compressed_artifact': str(artifact_path),
            'reconstruction_path': str(reconstruction_path),
        })
        print(
            f"{video_name} [{codec_name}] | BER={ber:.4f} | "
            f"size_ratio={size_ratio:.4f}"
        )

print("\nMetrik dasar selesai. Status deteksi dihitung dari uji binomial pada Bagian 10B.")


## 10. Evaluasi Metrik Kualitas & Kemiripan

Menghitung metrik evaluasi untuk setiap video dan setiap metode kompresi: **BER, PSNR, SSIM, Similarity Score, dan Compression Ratio**. Kelimanya menjadi masukan bagi CACS pada Bagian 10B, tempat **Watermark Recovery Rate (TPR)** dan status deteksi dihitung.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim
import pandas as pd

def cosine_similarity(a, b):
    a = np.array(a).flatten(); b = np.array(b).flatten()
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def compute_psnr_ssim(video_a_path, video_b_path):
    """Rata-rata PSNR & SSIM antar frame dua video (misal: watermarked vs compressed)."""
    frames_a, _, _ = extract_frames(video_a_path)
    frames_b, _, _ = extract_frames(video_b_path)
    n_compare = min(len(frames_a), len(frames_b))

    psnr_vals, ssim_vals = [], []
    for i in range(n_compare):
        fa, fb = frames_a[i], frames_b[i]
        psnr_vals.append(sk_psnr(fa, fb, data_range=1.0))
        ssim_vals.append(sk_ssim(fa, fb, data_range=1.0, channel_axis=-1))
    return float(np.mean(psnr_vals)), float(np.mean(ssim_vals))

eval_rows = []
for rec in verification_records:
    vname, codec_name = rec['video'], rec['codec']
    wm_path = watermarked_videos[vname]
    comp_path = compressed_paths[vname][codec_name]

    psnr_val, ssim_val = compute_psnr_ssim(wm_path, comp_path)
    bits_pred, probs = extraction_results[vname][codec_name]
    sim_score = cosine_similarity(gt_bits, bits_pred)

    eval_rows.append({
        'Video': vname,
        'Metode Kompresi': codec_name.upper(),
        'BER': round(rec['ber'], 4),
        'PSNR (dB)': round(psnr_val, 2),
        'SSIM': round(ssim_val, 4),
        'Similarity Score': round(sim_score, 4),
        'Compression Ratio': round(rec['compression_ratio'], 4),
        'ECC Text': probabilities_to_text_ecc(probs),
        'Exact Text Match': probabilities_to_text_ecc(probs) == WATERMARK_TEXT,
    })

df_eval = pd.DataFrame(eval_rows)

print("=== Tabel Metrik Evaluasi (sebelum keputusan CACS) ===")
display(df_eval)

# Simpan hasil evaluasi ke Google Drive
df_eval.to_csv(str(REPORTS_DIR / 'hasil_evaluasi_detail.csv'), index=False)
print("\nHasil evaluasi tersimpan di Google Drive:", OUTPUT_DIR)
print("Status Terdeteksi/Tidak Terdeteksi & Watermark Recovery Rate (TPR) dihitung pada Bagian 10B (CACS).")

## 10B. Metode Usulan (Novelty): **Compression-Aware Confidence Score (CACS)**

### 10B.1 Latar belakang masalah
Pada sistem watermarking konvensional, keputusan *Terdeteksi / Tidak Terdeteksi* diambil dari **satu metrik tunggal** (umumnya BER atau Similarity Score) yang dibandingkan terhadap sebuah ambang tetap. Cara ini memiliki dua kelemahan mendasar:

1. **Bukti tunggal.** Kualitas sinyal yang benar-benar sampai ke decoder (PSNR, SSIM) dan tingkat keparahan kompresi (Compression Ratio) tidak ikut diperhitungkan, padahal ketiganya menentukan seberapa layak dipercaya hasil ekstraksi bit.
2. **Ambang yang ditetapkan manual.** Nilai ambang seperti 0,20 atau 0,25 pada banyak implementasi tidak diturunkan dari dasar statistik apa pun.

**CACS** diusulkan untuk mengatasi keduanya: seluruh metrik yang **sudah dihitung sistem** (BER, PSNR, SSIM, Similarity Score, Compression Ratio) dinormalisasi terhadap *acceptance anchor* dari literatur, lalu digabung menjadi **satu skor kepercayaan** pada rentang [0, 1]. Bagian 9 dan 10 hanya bertugas **mengukur**; seluruh **keputusan** diambil di sini.

---

### 10B.2 Dasar ilmiah pemilihan tiap metrik

| Metrik | Peran dalam CACS | Dasar rujukan |
|---|---|---|
| **BER** | Bukti utama keberadaan watermark (*payload evidence*) | BER adalah tolok ukur baku akurasi ekstraksi watermark biner bersama NC (Elsevier, *Information Sciences*, Hsu & Tu, 2019; MDPI *Electronics* 12(19):4117, 2023) |
| **Similarity Score (NC/NCC)** | Bukti pendukung kemiripan pola bit hasil ekstraksi terhadap *ground truth* | NC ≥ 0,75 lazim dipakai sebagai kriteria "ekstraksi berhasil" (MDPI *Applied Sciences* 13(23):12957, 2023; *Scientific Reports* 15, 2025 memvalidasi ketahanan dengan NCC > 0,75). Untuk video anti-rekompresi H.264 dipakai NC > 0,8 (MDPI *Mathematics* 11(13):2913, 2023) |
| **PSNR** | Kualitas sinyal yang sampai ke decoder (*channel/evidence reliability*) | PSNR > 30 dB dinyatakan sebagai kualitas visual yang dapat diterima (*Scientific Reports* 14, s41598-024-76101-w, 2024; Jurnal Teknik SILITEK 5(3), 2025 menyatakan PSNR ≥ 30 dB) |
| **SSIM** | Integritas struktural frame setelah kompresi | SSIM mendekati 1 menandakan kualitas perseptual tinggi (*Scientific Reports* 14, 2024). Skema video watermarking robust melaporkan SSIM 0,99 (MDPI *Mathematics* 11(13):2913, 2023) dan 0,996 (MDPI *Mathematics* 13(15):2493, 2025) |
| **Compression Ratio** | Konteks *compression-aware*: seberapa berat tekanan kompresi yang dialami bukti | Rasio/laju bit adalah dimensi evaluasi standar pada watermarking video terkompresi; skema H.264 dilaporkan bertahan pada rasio kompresi hingga 40:1 (Wu et al., dirangkum dalam tesis Georgia Tech tentang *compressed-domain video watermarking*) |

---

### 10B.3 Normalisasi (tanpa angka arbitrer)

Setiap metrik dipetakan ke [0, 1] dengan **fungsi dua-segmen** yang menempatkan **nilai-terima menurut literatur tepat pada skor 0,5**:

$$
s(x)=egin{cases}
0, & x \le x_{low}\[2pt]
0.5\,\dfrac{x-x_{low}}{x_{acc}-x_{low}}, & x_{low}<x<x_{acc}\[6pt]
0.5+0.5\,\dfrac{x-x_{acc}}{x_{high}-x_{acc}}, & x_{acc}\le x<x_{high}\[6pt]
1, & x \ge x_{high}
\end{cases}
$$

| Metrik | $x_{low}$ (skor 0) | $x_{acc}$ (skor 0,5) | $x_{high}$ (skor 1) |
|---|---|---|---|
| BER | 0,5 (tebakan acak) | **hasil uji binomial** (lihat 10B.4) | 0 |
| PSNR | 20 dB | **30 dB** (literatur) | 45 dB |
| SSIM | 0,50 | **0,90** (literatur) | 1,00 |
| Similarity/NC | 0,00 | **0,75** (literatur) | 1,00 |
| Compression Ratio | $\log_{10}$CF = 2 (100×) | $\log_{10}$CF = 1 (**10×**) | $\log_{10}$CF = 0 (tanpa kompresi) |

CF (*compression factor*) $= 1/	ext{CR}$ dan diproses pada skala logaritmik karena laju bit bersifat multiplikatif, bukan aditif.

**Konsekuensi penting:** karena setiap $s_i = 0{,}5$ tepat pada kriteria terima literatur, maka ambang keputusan CACS $	au = 0{,}50$ **bukan angka pilihan bebas**, melainkan hasil konstruksi normalisasi itu sendiri.

---

### 10B.4 Ambang BER diturunkan secara statistik (bukan ditetapkan)

Deteksi watermark diperlakukan sebagai **uji hipotesis** (praktik baku pada literatur watermarking):

- $H_0$: video **tidak** mengandung watermark → tiap bit hasil ekstraksi bersifat Bernoulli(0,5).
- Jumlah bit cocok $K \sim 	ext{Binomial}(n, 0{,}5)$ dengan $n$ = `WATERMARK_LENGTH`.
- Dipilih $k_{min}$ terkecil sehingga $P(K \ge k_{min}\mid H_0)=\sum_{i=k_{min}}^{n}inom{n}{i}2^{-n} \le 	ext{FPR}_{target}$.
- $	ext{BER}_{thr} = (n-k_{min})/n$.

Dengan $n=64$ dan $	ext{FPR}_{target}=10^{-6}$ diperoleh $k_{min}=51$ sehingga $	ext{BER}_{thr}=0{,}2031$. Nilai ini **dihitung program**, bukan diketik manual, dan otomatis menyesuaikan bila panjang watermark diubah.

---

### 10B.5 Agregasi: equal weighting

Penelusuran literatur (IEEE/Springer/Elsevier/MDPI/SINTA) **tidak menemukan** satu pun rujukan yang memberikan bobot gabungan baku untuk kelima metrik ini — masing-masing selalu dilaporkan terpisah. Karena itu digunakan **equal weighting** ($w_i = 1/5$), yang menurut *OECD/JRC Handbook on Constructing Composite Indicators* (2008) merupakan pilihan yang sah dan paling lazim ketika tidak tersedia dasar teoretis untuk membedakan kepentingan antar-indikator, sekaligus menghindari asumsi *a priori* mengenai dimensi mana yang lebih dominan:

$$	ext{CACS}=rac{1}{5}\left(s_{BER}+s_{PSNR}+s_{SSIM}+s_{SIM}+s_{CR}
ight)$$

---

### 10B.6 Aturan keputusan dua-syarat

$$	ext{Detected} \iff ig(	ext{BER} \le 	ext{BER}_{thr}ig) \;\wedge\; ig(	ext{CACS} \ge 0{,}50ig)$$

- **Syarat perlu (statistik):** menjaga *false positive rate* $\le 10^{-6}$ sehingga skor tinggi pada PSNR/SSIM/CR **tidak dapat mengkompensasi** ketiadaan bukti payload — mencegah kelemahan klasik agregasi aditif (*compensability*, OECD/JRC 2008).
- **Syarat cukup (agregat):** memastikan secara keseluruhan sistem berada pada atau di atas kriteria terima literatur.

Tingkat kepercayaan dilaporkan sebagai: **Tinggi** (CACS ≥ 0,75), **Sedang** (0,50 ≤ CACS < 0,75), **Rendah** (CACS < 0,50).

---

### 10B.7 Posisi sebagai kontribusi penelitian

CACS adalah **metode yang diusulkan (*proposed method*)**. Yang bersifat kutipan literatur adalah *acceptance anchor* tiap metrik dan prosedur uji binomial; sedangkan **skema normalisasi dua-segmen, penggabungan lima metrik menjadi satu skor kepercayaan yang sadar-kompresi, dan aturan keputusan dua-syarat merupakan kontribusi orisinal penelitian ini.** CACS merupakan **satu-satunya lapisan pengambil keputusan** pada sistem ini — tidak ada ambang tetap maupun skema thresholding lain yang berjalan paralel.

**Keterbatasan yang perlu dinyatakan dalam naskah:** (i) `cosine_similarity` dihitung pada vektor bit {0,1} sehingga nilainya cenderung lebih tinggi daripada NCC berbasis {−1,+1}; (ii) PSNR/SSIM di sini mengukur distorsi *watermarked → compressed*, jadi merepresentasikan keparahan kanal, bukan imperceptibility penyisipan; (iii) equal weighting dapat diganti *entropy weighting* atau AHP bila tersedia data lebih besar — layak dijadikan saran penelitian lanjutan.

In [ ]:
from math import comb, log10
import numpy as np
import pandas as pd
import os

# ----------------------------------------------------------
# (1) ANCHOR DESKRIPTIF CACS (eksploratif; bukan classifier tervalidasi)
#     Format anchor: (x_low, x_acc, x_high)
#       x_low  = batas bawah (skor 0)
#       x_acc  = nilai "diterima" menurut literatur (skor 0.5)
#       x_high = kondisi ideal (skor 1)
# ----------------------------------------------------------
TARGET_FPR = 1e-6          # target probabilitas false alarm untuk uji binomial

PSNR_ANCHOR = (20.0, 30.0, 45.0)    # 30 dB = batas kualitas visual diterima
SSIM_ANCHOR = (0.50, 0.90, 1.00)    # SSIM mendekati 1 = kualitas perseptual tinggi
SIM_ANCHOR  = (0.00, 0.75, 1.00)    # NC/Similarity >= 0.75 = ekstraksi dianggap berhasil
LOGCF_ANCHOR = (2.0, 1.0, 0.0)      # log10(faktor kompresi): 100x, 10x, 1x (tanpa kompresi)

CACS_DECISION_THRESHOLD = 0.50      # titik netral hasil konstruksi normalisasi


def ber_threshold_from_fpr(n_bits, target_fpr=TARGET_FPR):
    """Ambang BER diturunkan dari uji hipotesis binomial.

    H0: video tidak ber-watermark -> tiap bit hasil ekstraksi ~ Bernoulli(0.5).
    Jumlah bit yang cocok K ~ Binomial(n, 0.5).
    Dipilih k_min terkecil sehingga P(K >= k_min | H0) <= target_fpr,
    lalu BER_thr = (n - k_min) / n.  Sepenuhnya non-arbitrer.
    """
    total = 2.0 ** n_bits
    tail, k_min = 0.0, n_bits
    for k in range(n_bits, -1, -1):
        tail += comb(n_bits, k)
        if tail / total <= target_fpr:
            k_min = k
        else:
            break
    return (n_bits - k_min) / n_bits, k_min


def norm_anchor(x, anchor, higher_is_better=True):
    """Normalisasi dua-segmen ke [0,1]; nilai 'diterima' menurut literatur -> 0.5."""
    x_low, x_acc, x_high = anchor
    if not higher_is_better:
        x, x_low, x_acc, x_high = -x, -x_low, -x_acc, -x_high
    if x <= x_low:
        return 0.0
    if x >= x_high:
        return 1.0
    if x < x_acc:
        return 0.5 * (x - x_low) / (x_acc - x_low)
    return 0.5 + 0.5 * (x - x_acc) / (x_high - x_acc)


# Ambang BER dihitung otomatis mengikuti panjang watermark yang dipakai sistem
BER_THR, K_MIN = ber_threshold_from_fpr(WATERMARK_LENGTH, TARGET_FPR)
BER_ANCHOR = (0.5, BER_THR, 0.0)    # 0.5 = tebakan acak (informasi nol)

print(f"[CACS] Panjang watermark            : {WATERMARK_LENGTH} bit")
print(f"[CACS] Target FPR (uji binomial)    : {TARGET_FPR:.0e}")
print(f"[CACS] Bit cocok minimum (k_min)    : {K_MIN}/{WATERMARK_LENGTH}")
print(f"[CACS] Ambang BER hasil turunan     : {BER_THR:.4f}  <- SATU-SATUNYA dasar 'Status Deteksi'")
print(f"[CACS] Ambang keputusan CACS        : {CACS_DECISION_THRESHOLD:.2f}  (skor kualitas/confidence, TIDAK menggerbang status deteksi)\n")


# ----------------------------------------------------------
# (2) FUNGSI INTI CACS
# ----------------------------------------------------------
def compute_cacs(ber, psnr, ssim, sim, compression_ratio):
    """Hitung Compression-Aware Confidence Score (CACS) + sub-skornya.

    compression_ratio = ukuran artefak codec / ukuran file video sumber
    (semakin kecil -> kompresi semakin berat).
    """
    cf = 1.0 / max(float(compression_ratio), 1e-9)      # faktor kompresi (mis. 20x)
    s_ber  = norm_anchor(ber,  BER_ANCHOR,  higher_is_better=False)
    s_psnr = norm_anchor(psnr, PSNR_ANCHOR, higher_is_better=True)
    s_ssim = norm_anchor(ssim, SSIM_ANCHOR, higher_is_better=True)
    s_sim  = norm_anchor(sim,  SIM_ANCHOR,  higher_is_better=True)
    s_cr   = norm_anchor(log10(max(cf, 1e-9)), LOGCF_ANCHOR, higher_is_better=False)

    subs = {'s_BER': s_ber, 's_PSNR': s_psnr, 's_SSIM': s_ssim,
            's_SIM': s_sim, 's_CR': s_cr}
    cacs = float(np.mean(list(subs.values())))          # equal weighting (w_i = 1/5)
    return cacs, subs


def cacs_decision(ber, cacs):
    """PERBAIKAN: 'Status Deteksi' sekarang MURNI berdasar uji statistik BER
    (BER <= BER_THR, turunan uji binomial dgn FPR terkendali) -- bukan lagi
    digabung dengan syarat CACS >= 0.50.

    Alasan: CACS menggabungkan s_BER (decodability) DENGAN s_PSNR/s_SSIM
    (kualitas visual video HASIL KOMPRESI). PSNR/SSIM video yang dikompres
    CRF tinggi secara ALAMI rendah (~20-27 dB) terlepas dari apakah watermark-nya
    terbaca benar atau tidak -- itu efek kompresi terhadap video apa pun, bukan
    indikator watermark gagal. Mensyaratkan CACS>=0.50 (yang ikut memuat
    s_PSNR/s_SSIM rendah itu) untuk status 'Terdeteksi' membuat status ini
    hampir selalu gagal justru saat kompresi paling berat -- padahal itu skenario
    yang seharusnya jadi bukti keberhasilan sistem (judul skripsi: tahan
    KOMPRESI TINGGI), bukan sumber kegagalan otomatis.

    CACS (dengan s_PSNR/s_SSIM di dalamnya) tetap dihitung & dilaporkan penuh
    di bawah -- sebagai SKOR KUALITAS/CONFIDENCE terpisah, bukan syarat wajib
    status deteksi.
    """
    return bool(ber <= BER_THR)


def cacs_confidence_level(cacs):
    if cacs >= 0.75:
        return 'Tinggi'
    if cacs >= CACS_DECISION_THRESHOLD:
        return 'Sedang'
    return 'Rendah'


# ----------------------------------------------------------
# (3) TERAPKAN KE SELURUH HASIL EVALUASI (df_eval dari Bagian 10)
# ----------------------------------------------------------
# PERBAIKAN: bikin sel ini AMAN dijalankan berulang kali tanpa mengulang dari
# Bagian 10. Sebelumnya, menjalankan ulang sel ini menambahkan set kolom
# s_BER/CACS/Status Deteksi dkk LAGI ke df_eval yang sudah punya kolom itu dari
# run sebelumnya -> muncul 2 kolom bernama sama -> df_eval['Status Deteksi']
# malah balikin DataFrame (bukan Series) -> ValueError saat groupby/assign.
_cacs_col_names = ['s_BER', 's_PSNR', 's_SSIM', 's_SIM', 's_CR',
                    'CACS', 'Confidence', 'Status Deteksi']
df_eval = df_eval.drop(columns=[c for c in _cacs_col_names if c in df_eval.columns])

_cacs_cols = []
for _, row in df_eval.iterrows():
    cacs, subs = compute_cacs(row['BER'], row['PSNR (dB)'], row['SSIM'],
                              row['Similarity Score'], row['Compression Ratio'])
    detected = cacs_decision(row['BER'], cacs)
    _cacs_cols.append({
        **{k: round(v, 4) for k, v in subs.items()},
        'CACS': round(cacs, 4),
        'Confidence': cacs_confidence_level(cacs),
        'Status Deteksi': 'Terdeteksi' if detected else 'Tidak Terdeteksi',
    })

df_eval = pd.concat([df_eval.reset_index(drop=True),
                     pd.DataFrame(_cacs_cols)], axis=1)

# Semua sampel di tabel ini positif/ber-watermark, jadi metrik yang benar adalah recovery rate (TPR/sensitivity), bukan classification accuracy.
accuracy_per_codec = (
    df_eval.assign(is_detected=df_eval['Status Deteksi'] == 'Terdeteksi')
           .groupby('Metode Kompresi')['is_detected']
           .mean()
           .rename('Watermark Recovery Rate (TPR)')
           .reset_index()
)

cacs_per_codec = (
    df_eval.groupby('Metode Kompresi')['CACS'].mean()
           .rename('Rata-rata CACS').reset_index()
)

print("=== Tabel Evaluasi + CACS ===")
display(df_eval[['Video', 'Metode Kompresi', 'BER', 'PSNR (dB)', 'SSIM',
                 'Similarity Score', 'Compression Ratio',
                 's_BER', 's_PSNR', 's_SSIM', 's_SIM', 's_CR',
                 'CACS', 'Confidence', 'Status Deteksi']])

print("\n=== Watermark Recovery Rate (TPR) & Rata-rata CACS per Codec ===")
display(accuracy_per_codec.merge(cacs_per_codec, on='Metode Kompresi'))

df_eval.to_csv(str(REPORTS_DIR / 'hasil_evaluasi_cacs.csv'), index=False)
accuracy_per_codec.to_csv(str(REPORTS_DIR / 'hasil_watermark_recovery_rate.csv'), index=False)
print("\nHasil CACS tersimpan di:", OUTPUT_DIR)

## 11. Visualisasi & Analisis Hasil

Menampilkan grafik perbandingan performa sistem watermarking pada ketiga metode kompresi (H.264, H.265, Neural Codec).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# (a) Watermark recovery rate / true-positive rate
axes[0, 0].bar(accuracy_per_codec['Metode Kompresi'], accuracy_per_codec['Watermark Recovery Rate (TPR)'], color='#4C72B0')
axes[0, 0].set_title('Watermark Recovery Rate (TPR) per Codec')
axes[0, 0].set_ylim(0, 1.05)

# (b) Rata-rata BER
ber_mean = df_eval.groupby('Metode Kompresi')['BER'].mean()
axes[0, 1].bar(ber_mean.index, ber_mean.values, color='#DD8452')
axes[0, 1].set_title('Rata-rata Bit Error Rate (BER)')

# (c) Rata-rata PSNR
psnr_mean = df_eval.groupby('Metode Kompresi')['PSNR (dB)'].mean()
axes[1, 0].bar(psnr_mean.index, psnr_mean.values, color='#55A868')
axes[1, 0].set_title('Rata-rata PSNR (dB)')

# (d) Rata-rata SSIM
ssim_mean = df_eval.groupby('Metode Kompresi')['SSIM'].mean()
axes[1, 1].bar(ssim_mean.index, ssim_mean.values, color='#8172B2')
axes[1, 1].set_title('Rata-rata SSIM')

plt.tight_layout()
plot_path = str(REPORTS_DIR / 'grafik_analisis_hasil.png')
plt.savefig(plot_path, dpi=150)
plt.show()

print("Grafik analisis tersimpan di:", plot_path)

# ==========================================================
# 12. DETEKSI WATERMARK SEMUA FRAME + VISUALISASI POSISI WATERMARK INVISIBLE
# ==========================================================
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as patches

encoder.eval()
decoder.eval()

# --- Pilih video yang mau dianalisis ---
video_name = list(watermarked_videos.keys())[0]      # video pertama di dataset
codec_used = 'h264'                              # bisa ganti 'h265' atau 'neural'
target_compressed_path = compressed_paths[video_name][codec_used]

original_frames, fps_orig, _ = dataset_frames[video_name]
n_frames = len(original_frames)
print(f"Menganalisis video: {video_name} | total frame: {n_frames} | metode kompresi: {codec_used}")

# ==========================================================
# A. HITUNG RESIDUAL WATERMARK (POSISI WATERMARK INVISIBLE) PER FRAME
# ==========================================================
orig_t = torch.tensor(np.stack(original_frames)).permute(0, 3, 1, 2).float().to(device)
wm_batch = GROUND_TRUTH_WM.repeat(orig_t.size(0), 1)

with torch.no_grad():
    watermarked_t, residual_t = encoder(orig_t, wm_batch)

residual_np = residual_t.permute(0, 2, 3, 1).cpu().numpy()          # (N, H, W, 3), nilai kecil (~±0.05)
watermarked_np = watermarked_t.permute(0, 2, 3, 1).cpu().numpy()

# CATATAN PENTING (untuk sidang):
# Encoder ini menyisipkan watermark secara TERSEBAR ke seluruh frame (bukan di satu blok/kotak
# tertentu seperti watermarking spasial klasik) -- ini sengaja, agar watermark tetap bisa
# diekstrak walau sebagian frame terpotong/rusak akibat kompresi. Karena itu, "posisi" watermark
# di sini dimaknai sebagai ZONA KONSENTRASI TERKUAT (wilayah residual watermark paling besar),
# bukan lokasi tunggal tempat seluruh informasi watermark berada.

def residual_to_heatmap(residual):
    """Normalisasi RELATIF per-frame (bukan skala tetap) supaya variasi intensitas
    watermark benar-benar kelihatan, bukan blok warna yang jenuh/rata."""
    mag = np.abs(residual).sum(axis=-1)                      # gabung 3 channel jadi 1 peta intensitas
    mag_norm = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    heatmap = cm.inferno(mag_norm)[..., :3]                  # colormap 'inferno' -> terang = watermark lebih kuat
    return heatmap, mag_norm

def compute_strongest_zone_bbox(mag_norm, top_percent=15, min_size=12):
    """Cari bounding box wilayah dengan intensitas watermark TERKUAT (top N% piksel).
    Ini adalah 'zona penyisipan paling dominan', bukan satu-satunya lokasi watermark."""
    thresh = np.percentile(mag_norm, 100 - top_percent)
    mask = mag_norm >= thresh
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    # jaga agar kotak tidak terlalu kecil supaya tetap terlihat jelas
    if (x1 - x0) < min_size:
        cx = (x0 + x1) // 2
        x0, x1 = max(0, cx - min_size // 2), min(mag_norm.shape[1] - 1, cx + min_size // 2)
    if (y1 - y0) < min_size:
        cy = (y0 + y1) // 2
        y0, y1 = max(0, cy - min_size // 2), min(mag_norm.shape[0] - 1, cy + min_size // 2)
    return (x0, y0, x1, y1)

# ==========================================================
# B. DETEKSI WATERMARK PER FRAME PADA VIDEO HASIL KOMPRESI
# ==========================================================
comp_frames, fps_comp, _ = extract_frames(target_compressed_path, max_frames=n_frames)
comp_frames_t = torch.tensor(np.stack(comp_frames)).permute(0, 3, 1, 2).float().to(device)

with torch.no_grad():
    logits = decoder(comp_frames_t)
    probs = torch.sigmoid(logits)
    bits_pred_per_frame = (probs > 0.5).float().cpu().numpy()

gt = np.array(gt_bits).flatten() if 'gt_bits' in globals() else GROUND_TRUTH_WM.cpu().numpy().flatten()

# PERBAIKAN: sebelumnya di sini ada BER_THRESHOLD = 0.25 yang di-hardcode terpisah
# dari BER_THR hasil uji binomial di Bagian 10B (CACS) -- artinya ada DUA aturan
# keputusan berbeda berjalan paralel dalam satu notebook (0.25 di sini vs ~0.2031
# di CACS), padahal CACS diklaim sebagai "satu-satunya lapisan pengambil keputusan".
# Sekarang cell ini memakai BER_THR yang SAMA (dihitung sekali di Bagian 10B),
# supaya status per-frame di sini dan status per-video di tabel CACS konsisten.
BER_THRESHOLD = BER_THR if 'BER_THR' in globals() else 0.25
frame_status = []
for i in range(bits_pred_per_frame.shape[0]):
    ber = np.mean(gt != bits_pred_per_frame[i])
    bit_acc = 1.0 - ber                                        # skor akurasi bit per frame
    status = 'Terdeteksi' if ber <= BER_THRESHOLD else 'Tidak Terdeteksi'
    decoded_text_i = probabilities_to_text_ecc(probs[i].detach().cpu().numpy())       # BUKTI isi watermark: decode balik jadi teks
    frame_status.append({
        'frame': i + 1,
        'ber': round(float(ber), 4),
        'akurasi_bit': round(float(bit_acc), 4),
        'status': status,
        'teks_terekstrak': decoded_text_i,
    })

df_status = pd.DataFrame(frame_status)
print(f"\nRingkasan deteksi {len(df_status)} frame:")
print(f" - Terdeteksi     : {(df_status['status']=='Terdeteksi').sum()}")
print(f" - Tidak Terdeteksi: {(df_status['status']=='Tidak Terdeteksi').sum()}")
display(df_status)

# ==========================================================
# C. VISUALISASI: FRAME ASLI | WATERMARKED + KOTAK ZONA TERKUAT | HEATMAP + KOTAK | STATUS DETEKSI
# ==========================================================
def show_watermark_position(n_show=8, save_to_drive=True, top_percent=15):
    n_show = min(n_show, n_frames)
    step = max(1, n_frames // n_show)
    idxs = list(range(0, n_frames, step))[:n_show]

    save_dir = os.path.join(OUTPUT_DIR, 'frames_tmp', f'{video_name}_watermark_position')
    if save_to_drive:
        os.makedirs(save_dir, exist_ok=True)

    fig, axes = plt.subplots(len(idxs), 4, figsize=(16, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = axes.reshape(1, -1)

    col_titles = ['Frame Asli', 'Frame Ber-Watermark + Zona Terkuat', 'Peta Intensitas Watermark + Kotak', 'Status Deteksi']
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=11)

    for row, idx in enumerate(idxs):
        heatmap, mag_norm = residual_to_heatmap(residual_np[idx])
        bbox = compute_strongest_zone_bbox(mag_norm, top_percent=top_percent)
        status_i = df_status.iloc[idx]['status'] if idx < len(df_status) else 'N/A'
        ber_i = df_status.iloc[idx]['ber'] if idx < len(df_status) else None
        acc_i = df_status.iloc[idx]['akurasi_bit'] if idx < len(df_status) else None
        teks_i = df_status.iloc[idx]['teks_terekstrak'] if idx < len(df_status) else ''
        bits_i = bits_pred_per_frame[idx].astype(int)
        bits_str = ''.join(bits_i.astype(str))
        bits_str_wrapped = '\n'.join(bits_str[j:j+16] for j in range(0, len(bits_str), 16))

        axes[row, 0].imshow(original_frames[idx]); axes[row, 0].axis('off')

        axes[row, 1].imshow(np.clip(watermarked_np[idx], 0, 1)); axes[row, 1].axis('off')
        axes[row, 2].imshow(heatmap); axes[row, 2].axis('off')

        if bbox is not None:
            x0, y0, x1, y1 = bbox
            for col in (1, 2):
                rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                          linewidth=2, edgecolor='lime', facecolor='none')
                axes[row, col].add_patch(rect)

        color = 'green' if status_i == 'Terdeteksi' else 'red'
        match_color = 'green' if teks_i == WATERMARK_TEXT else 'orange'
        axes[row, 3].text(0.5, 0.97, f'Frame {idx+1}', ha='center', va='top', fontsize=12)
        axes[row, 3].text(0.5, 0.88, status_i, ha='center', va='top', fontsize=13, color=color, fontweight='bold')
        axes[row, 3].text(0.5, 0.78, f'Skor Akurasi Bit: {acc_i:.2%}  |  BER: {ber_i}', ha='center', va='top', fontsize=9)
        axes[row, 3].text(0.5, 0.68, f"Teks Asli    : '{WATERMARK_TEXT}'", ha='center', va='top', fontsize=9)
        axes[row, 3].text(0.5, 0.60, f"Teks Terekstrak: '{teks_i}'", ha='center', va='top', fontsize=9,
                           color=match_color, fontweight='bold')
        axes[row, 3].text(0.5, 0.48, 'Bit Watermark Terekstrak (144 bit ECC):', ha='center', va='top', fontsize=8)
        axes[row, 3].text(0.5, 0.40, bits_str_wrapped, ha='center', va='top', fontsize=8, family='monospace')
        axes[row, 3].axis('off')

        if save_to_drive:
            combined = np.concatenate([
                original_frames[idx],
                np.clip(watermarked_np[idx], 0, 1),
                heatmap
            ], axis=1)
            out_path = os.path.join(save_dir, f'watermark_position_frame_{idx+1:04d}.png')
            plt.imsave(out_path, combined)

    plt.suptitle(f'Posisi Watermark Invisible — {video_name} (kompresi: {codec_used.upper()})\n'
                 f'Kotak hijau = zona konsentrasi watermark terkuat (top {top_percent}% intensitas) — '
                 f'watermark sesungguhnya tersebar di seluruh frame untuk ketahanan',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

    if save_to_drive:
        print(f"\nGambar posisi watermark (semua frame yang ditampilkan) tersimpan di: {save_dir}")

show_watermark_position(n_show=8, save_to_drive=True, top_percent=15)

## 12. Kesimpulan Penggunaan

Hasil eksperimen boleh dipakai hanya jika:

- `POST-V3 SANITY` menghasilkan `passed: True`;
- negative control tidak menyerupai `"sabila"`;
- random 144-bit dan random-ECC ikut terbaca, bukan hanya target produksi;
- tabel akhir berasal dari `eval_video_names` (group hold-out);
- neural compression ratio memakai file `*_neuralcodec_latent.npz`.

Jika gate gagal, naikkan `FT3_EPOCHS` atau jumlah video training lalu latih ulang.
Jangan mengaktifkan kembali fine-tune dengan target tetap `"sabila"`, karena itu dapat
menghasilkan BER semu 0.0 melalui decoder collapse.

Untuk laporan skripsi, sajikan sekurang-kurangnya: raw BER, exact-text recovery setelah
ECC, false-positive negative control, PSNR, SSIM, size ratio, dan hasil terpisah untuk
H.264, H.265, serta neural codec.
